<a href="https://colab.research.google.com/github/GokhanMurali/learn2branchbyGAT/blob/main/Experiment_2_Combinatorial_Auction_Problems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Branching by GAT: Combinatorial Auction Problems

In this notebook, I will do my experimentation on Combinatorial Auction problems.

### Part 1: Installing and Importing Libraries

In [ ]:
# to install required libraries, I will use conda
!pip install -q condacolab
import condacolab
condacolab.install()

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/23.11.0-0/Mambaforge-23.11.0-0-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:10
🔁 Restarting kernel...


In [ ]:
# let's check conda installed or not
!conda --version

conda 23.11.0


In [ ]:
# i will install ecole library
!conda install -c conda-forge ecole

Channels:
 - conda-forge
Platform: linux-64
Solving environment: - \ | / done


==> WARNING: A newer version of conda exists. <==
    current version: 23.11.0
    latest version: 24.11.0

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - ecole


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ampl-mp-3.1.0              |    h2cc385e_1006         1.1 MB  conda-forge
    ca-certificates-2024.8.30  |       hbcca054_0         155 KB  conda-forge
    certifi-2024.8.30          |     pyhd8ed1ab_0         160 KB  conda-forge
    cppad-20240000.2           |       h59595ed_0         483 KB  conda-forge
    ecole-0.8.1                |  py310h4c677e1_3         298 KB  conda-forge
    gmp-6.3.0                  |       hac33072_2         449 KB  conda-forge
    ipopt-3.14

In [ ]:
# i will install pytorch geometric
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.2/69.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.3/287.3 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.6/179.6 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.3/133.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.9/106.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.9/241.9 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.6/124.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.1/205.1 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# i will import required libraries
# hata vermemesi için usr/local/lib altındaki ilgili scip dosyasının 8.1.0.0 olan kısmını 8.0 olarak revize ediyorum
import csv
import gzip
import pickle
from pathlib import Path

import ecole
import numpy as np
import torch
import torch.nn.functional as F
import torch_geometric
from torch_geometric.data import HeteroData
from torch_geometric.nn import GATConv
from torch_geometric.nn import to_hetero
from torch_geometric.nn import Linear
from torch_geometric.nn import LayerNorm
from torch_geometric.nn import HeteroConv

### Part 2: Data Collection

In this part, I will generate training data from randomly generated 100.000 Set Covering Problem instances.

In [ ]:
# parameters
DATA_MAX_SAMPLES = 100000
LEARNING_RATE = 0.001
NB_EPOCHS = 50
NB_EVAL_INSTANCES = 100

I will use the Ecole-provided set cover instance generator.

In [ ]:
instances = ecole.instance.CombinatorialAuctionGenerator(n_items=100, n_bids=500, min_value=1, max_value=100,
                       value_deviation=0.5, add_item_prob=0.65, max_n_sub_bids=5,
                       additivity=0.2, budget_factor=1.5, resale_factor=0.5,
                       integers=False, warnings=False)

I will use following class to observe Strong Branching scores and bipartite graph features. In this scheme, to diversify the states in which we collect examples of strong branching behavior, we mostly follow a weak but cheap expert (pseudocost branching) and only occasionally call the strong expert (strong branching). This also ensures that samples are closer to being independent and identically distributed.

In [ ]:
class ExploreThenStrongBranch:
    """
    This custom observation function class will randomly return either strong branching scores (expensive expert)
    or pseudocost scores (weak expert for exploration) when called at every node.
    """

    def __init__(self, expert_probability):
        self.expert_probability = expert_probability
        self.pseudocosts_function = ecole.observation.Pseudocosts()
        self.strong_branching_function = ecole.observation.StrongBranchingScores()

    def before_reset(self, model):
        """
        This function will be called at initialization of the environment (before dynamics are reset).
        """
        self.pseudocosts_function.before_reset(model)
        self.strong_branching_function.before_reset(model)

    def extract(self, model, done):
        """
        Should we return strong branching or pseudocost scores at time node?
        """
        probabilities = [1 - self.expert_probability, self.expert_probability]
        expert_chosen = bool(np.random.choice(np.arange(2), p=probabilities))
        if expert_chosen:
            return (self.strong_branching_function.extract(model, done), True)
        else:
            return (self.pseudocosts_function.extract(model, done), False)

We can now create the environment with the correct parameters (no restarts, 1h time limit, 5% expert sampling probability).

In [ ]:
# We can pass custom SCIP parameters easily
scip_parameters = {
    "separating/maxrounds": 0,
    "presolving/maxrestarts": 0,
    "limits/time": 3600,
}

# Note how we can tuple observation functions to return complex state information
env = ecole.environment.Branching(
    observation_function=(
        ExploreThenStrongBranch(expert_probability=0.05),
        ecole.observation.NodeBipartite(),
    ),
    information_function={
        "nb_nodes": ecole.reward.NNodes(),
    },
    scip_params=scip_parameters,
)

# This will seed the environment for reproducibility
env.seed(0)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Now we loop over the instances, following the strong branching expert 5% of the time and saving its decision, until enough samples are collected.

In [ ]:
episode_counter, sample_counter = 0, 0
Path("drive/MyDrive/Thesis/ca_samples/").mkdir(exist_ok=True)

main_node_list = []
# We will solve problems (run episodes) until we have saved enough samples
while sample_counter < DATA_MAX_SAMPLES:
    episode_counter += 1
    observation, action_set, _, done, info = env.reset(next(instances))
    nb_nodes = 0
    node_list = []

    while not done:
        (scores, scores_are_expert), node_observation = observation
        action = action_set[scores[action_set].argmax()]
        nb_nodes += info["nb_nodes"]

        # Only save samples if they are coming from the expert (strong branching)
        if scores_are_expert and (sample_counter < DATA_MAX_SAMPLES):
            sample_counter += 1
            data = [node_observation, action, action_set, scores]
            filename = f"drive/MyDrive/Thesis/ca_samples/sample_{sample_counter}.pkl"
            node_list.append(int(nb_nodes))

            constraint_features = node_observation.row_features
            edge_indices = node_observation.edge_features.indices.astype(np.int32)
            edge_features = np.expand_dims(node_observation.edge_features.values, axis=-1)
            variable_features = node_observation.variable_features

            with gzip.open(filename, "wb") as f:
                pickle.dump(data, f)

        observation, action_set, _, done, info = env.step(action)

    nb_nodes += info["nb_nodes"]
    node_list.append(int(nb_nodes))
    print(f"Episode {episode_counter}, {sample_counter} samples collected so far")
    main_node_list.append(node_list)
with open("file_name.csv", 'w') as f:
    fc = csv.writer(f, lineterminator='\n')
    fc.writerows(main_node_list)

Episode 1, 0 samples collected so far
Episode 2, 0 samples collected so far


In [ ]:
# zip the sample files
!zip -r /content/drive/MyDrive/Thesis/ca_samples.zip /content/drive/MyDrive/Thesis/ca_samples

Görüntülenen çıkış son 5000 satıra kısaltıldı.
  adding: content/drive/MyDrive/Thesis/ca_samples/sample_4002.pkl (deflated 0%)
  adding: content/drive/MyDrive/Thesis/ca_samples/sample_4003.pkl (deflated 0%)
  adding: content/drive/MyDrive/Thesis/ca_samples/sample_4004.pkl (deflated 0%)
  adding: content/drive/MyDrive/Thesis/ca_samples/sample_4005.pkl (deflated 0%)
  adding: content/drive/MyDrive/Thesis/ca_samples/sample_4006.pkl (deflated 0%)
  adding: content/drive/MyDrive/Thesis/ca_samples/sample_4007.pkl (deflated 0%)
  adding: content/drive/MyDrive/Thesis/ca_samples/sample_4008.pkl (deflated 0%)
  adding: content/drive/MyDrive/Thesis/ca_samples/sample_4009.pkl (deflated 0%)
  adding: content/drive/MyDrive/Thesis/ca_samples/sample_4010.pkl (deflated 0%)
  adding: content/drive/MyDrive/Thesis/ca_samples/sample_4011.pkl (deflated 0%)
  adding: content/drive/MyDrive/Thesis/ca_samples/sample_4012.pkl (deflated 0%)
  adding: content/drive/MyDrive/Thesis/ca_samples/sample_4013.pkl (deflat

In [ ]:
# download the zipped files
from google.colab import files
files.download("/content/file.zip")

FileNotFoundError: Cannot find file: /content/file.zip

Let's check some data.

In [ ]:
# (X, 5) shape
print(constraint_features)
print(constraint_features.shape)

NameError: name 'constraint_features' is not defined

In [ ]:
# (2, Y) shape
print(edge_indices)
print(edge_indices.shape)

In [ ]:
# (Y, 1) shape
print(edge_features)
print(edge_features.shape)

In [ ]:
# action = branching variable
print(data[1])
# action set = branching candidates
print(data[2])
# strong branching scores
print(data[3])
# action
print(data[2][data[3][data[2]].argmax()])

### Part 3: Generating Datasets

We will first define pytorch geometric data classes to handle the bipartite graph data.

In [ ]:
class BipartiteNodeData(torch_geometric.data.Data):
    """
    This class encode a node bipartite graph observation as returned by the `ecole.observation.NodeBipartite`
    observation function in a format understood by the pytorch geometric data handlers.
    """

    def __init__(
        self,
        constraint_features,
        edge_indices,
        edge_features,
        variable_features,
        candidates,
        nb_candidates,
        candidate_choice,
        candidate_scores,
    ):
        super().__init__()
        self.constraint_features = constraint_features
        self.edge_index = edge_indices
        self.edge_attr = edge_features
        self.variable_features = variable_features
        self.candidates = candidates
        self.nb_candidates = nb_candidates
        self.candidate_choices = candidate_choice
        self.candidate_scores = candidate_scores

    def __inc__(self, key, value, store, *args, **kwargs):
        """
        We overload the pytorch geometric method that tells how to increment indices when concatenating graphs
        for those entries (edge index, candidates) for which this is not obvious.
        """
        if key == "edge_index":
            return torch.tensor(
                [[self.constraint_features.size(0)], [self.variable_features.size(0)]]
            )
        elif key == "candidates":
            return self.variable_features.size(0)
        else:
            return super().__inc__(key, value, *args, **kwargs)


class GraphDataset(torch_geometric.data.Dataset):
    """
    This class encodes a collection of graphs, as well as a method to load such graphs from the disk.
    It can be used in turn by the data loaders provided by pytorch geometric.
    """

    def __init__(self, sample_files):
        super().__init__(root=None, transform=None, pre_transform=None)
        self.sample_files = sample_files

    def len(self):
        return len(self.sample_files)

    def get(self, index):
        """
        This method loads a node bipartite graph observation as saved on the disk during data collection.
        """
        with gzip.open(self.sample_files[index], "rb") as f:
            sample = pickle.load(f)

        sample_observation, sample_action, sample_action_set, sample_scores = sample

        constraint_features = sample_observation.row_features
        edge_indices = sample_observation.edge_features.indices.astype(np.int32)
        edge_features = np.expand_dims(sample_observation.edge_features.values, axis=-1)
        variable_features = sample_observation.variable_features

        # We note on which variables we were allowed to branch, the scores as well as the choice
        # taken by strong branching (relative to the candidates)
        candidates = np.array(sample_action_set, dtype=np.int32)
        candidate_scores = np.array([sample_scores[j] for j in candidates])
        candidate_choice = np.where(candidates == sample_action)[0][0]

        graph = BipartiteNodeData(
            torch.FloatTensor(constraint_features),
            torch.LongTensor(edge_indices),
            torch.FloatTensor(edge_features),
            torch.FloatTensor(variable_features),
            torch.LongTensor(candidates),
            len(candidates),
            torch.LongTensor([candidate_choice]),
            torch.FloatTensor(candidate_scores)
        )

        # We must tell pytorch geometric how many nodes there are, for indexing purposes
        graph.num_nodes = constraint_features.shape[0] + variable_features.shape[0]

        return graph

In [ ]:
!unzip -qq 'drive/MyDrive/Thesis/ca_samples.zip'

We can then prepare the data loaders.

In [ ]:
sample_files = [str(path) for path in Path("content/drive/MyDrive/Thesis/ca_samples/").glob("sample_*.pkl")]
train_files = sample_files[: int(0.8 * len(sample_files))]
valid_files = sample_files[int(0.8 * len(sample_files)) :]

train_data = GraphDataset(train_files)
train_loader = torch_geometric.loader.DataLoader(train_data, batch_size=32, shuffle=True)
valid_data = GraphDataset(valid_files)
valid_loader = torch_geometric.loader.DataLoader(valid_data, batch_size=128, shuffle=False)

In [ ]:
print(len(train_files))
print(len(valid_files))

80000
20000


I will generate HeteroGraphDataset for GAT.

In [ ]:
class MyHeteroData(torch_geometric.data.HeteroData):

    def __init__(
        self,
        variable_features
    ):
        super().__init__()
        self.variable_features = variable_features


    def __inc__(self, key, value, *args, **kwargs):
        if key == 'candidates':
            return self.variable_features.size(0)
        return super().__inc__(key, value, *args, **kwargs)

class HeteroGraphDataset(torch_geometric.data.Dataset):
    """
    This class encodes a collection of graphs, as well as a method to load such graphs from the disk.
    It can be used in turn by the data loaders provided by pytorch geometric.
    """

    def __init__(self, sample_files):
        super().__init__(root=None, transform=None, pre_transform=None)
        self.sample_files = sample_files

    def len(self):
        return len(self.sample_files)

    def get(self, index):
        """
        This method loads a node bipartite graph observation as saved on the disk during data collection.
        """
        with gzip.open(self.sample_files[index], "rb") as f:
            sample = pickle.load(f)

        sample_observation, sample_action, sample_action_set, sample_scores = sample

        graph = MyHeteroData(torch.FloatTensor(sample_observation.variable_features))

        #node features
        graph['constraint'].x = torch.FloatTensor(sample_observation.row_features)
        graph['variable'].x = torch.FloatTensor(sample_observation.variable_features)

        #edge indices
        graph['variable', 'participates', 'constraint'].edge_index = torch.stack([torch.LongTensor(sample_observation.edge_features.indices.astype(np.int32)[1]), torch.LongTensor(sample_observation.edge_features.indices.astype(np.int32)[0])], dim=0)
        graph['constraint', 'contains', 'variable'].edge_index = torch.LongTensor(sample_observation.edge_features.indices.astype(np.int32))

        #edge features
        graph['variable', 'participates', 'constraint'].edge_attr = torch.FloatTensor(np.expand_dims(sample_observation.edge_features.values, axis=-1))
        graph['constraint', 'contains', 'variable'].edge_attr = torch.FloatTensor(np.expand_dims(sample_observation.edge_features.values, axis=-1))

        #graph features
        graph['candidates'] = torch.LongTensor(np.array(sample_action_set, dtype=np.int32))
        graph['candidate_scores'] = torch.FloatTensor(np.array([sample_scores[j] for j in np.array(sample_action_set, dtype=np.int32)]))
        graph['candidate_choices'] = torch.LongTensor([np.where(np.array(sample_action_set, dtype=np.int32) == sample_action)[0][0]])
        graph['nb_candidates'] = len(np.array(sample_action_set, dtype=np.int32))

        return graph

I will generate hetero datasets for GAT.

In [ ]:
hetero_train_data = HeteroGraphDataset(train_files)
hetero_train_loader = torch_geometric.loader.DataLoader(hetero_train_data, batch_size=32, shuffle=True)
hetero_valid_data = HeteroGraphDataset(valid_files)
hetero_valid_loader = torch_geometric.loader.DataLoader(hetero_valid_data, batch_size=32, shuffle=False)

In [ ]:
hetero_batch_list = []
for hetero_batch in hetero_train_loader:
    hetero_batch_list.append(hetero_batch)
    if len(hetero_batch_list)>1:
      break

In [ ]:
hetero_batch_list[0]

MyHeteroDataBatch(
  variable_features=[12929, 19],
  candidates=[2179],
  candidate_scores=[2179],
  candidate_choices=[32],
  nb_candidates=[32],
  constraint={
    x=[6916, 5],
    batch=[6916],
    ptr=[33],
  },
  variable={
    x=[12929, 19],
    batch=[12929],
    ptr=[33],
  },
  (variable, participates, constraint)={
    edge_index=[2, 129193],
    edge_attr=[129193, 1],
  },
  (constraint, contains, variable)={
    edge_index=[2, 129193],
    edge_attr=[129193, 1],
  }
)

In [ ]:
hetero_batch_list[0].candidates[70:80]

tensor([358, 359, 360, 363, 368, 380, 385, 389, 392, 398])

In [ ]:
hetero_batch_list[0].nb_candidates

tensor([77, 73, 67, 83, 68, 41, 84, 99, 86, 78, 49, 46, 79, 44, 55, 73, 65, 37,
        59, 57, 50, 68, 79, 67, 66, 79, 68, 75, 78, 82, 72, 75])

In [ ]:
hetero_batch_list[0]['variable'].ptr

tensor([    0,   389,   789,  1221,  1641,  1998,  2364,  2757,  3191,  3599,
         4005,  4409,  4831,  5244,  5671,  6090,  6522,  6935,  7294,  7717,
         8113,  8500,  8900,  9326,  9717, 10151, 10563, 10945, 11373, 11748,
        12172, 12563, 12929])

In [ ]:
hetero_batch_list[0]['variable', 'participates', 'constraint'].edge_index

tensor([[   26,    78,   110,  ..., 12871, 12907, 12919],
        [    0,     0,     0,  ...,  6915,  6915,  6915]])

In [ ]:
hetero_batch_list[0]['constraint', 'contains', 'variable'].edge_index

tensor([[    0,     0,     0,  ...,  6915,  6915,  6915],
        [   26,    78,   110,  ..., 12871, 12907, 12919]])

Let's check some data.

In [ ]:
print(valid_data[8])
print(hetero_valid_data[8])

BipartiteNodeData(constraint_features=[216, 5], edge_index=[2, 4001], edge_attr=[4001, 1], variable_features=[399, 19], candidates=[77], nb_candidates=77, candidate_choices=[1], candidate_scores=[77], num_nodes=615)
MyHeteroData(
  variable_features=[399, 19],
  candidates=[77],
  candidate_scores=[77],
  candidate_choices=[1],
  nb_candidates=77,
  constraint={ x=[216, 5] },
  variable={ x=[399, 19] },
  (variable, participates, constraint)={
    edge_index=[2, 4001],
    edge_attr=[4001, 1],
  },
  (constraint, contains, variable)={
    edge_index=[2, 4001],
    edge_attr=[4001, 1],
  }
)


In [ ]:
print(valid_data[9]['candidates'])
print(hetero_valid_data[9]['candidates'])
print(valid_data[9]['candidate_scores'])
print(hetero_valid_data[9]['candidate_scores'])
print(valid_data[9]['candidate_choices'])
print(hetero_valid_data[9]['candidate_choices'])
print(valid_data[9]['constraint_features'])
print(hetero_valid_data[9]['constraint'])
print(valid_data[9]['variable_features'])
print(hetero_valid_data[9]['variable'])
print(valid_data[9]['edge_index'])
print(valid_data[9]['edge_attr'])
print(hetero_valid_data[9]['constraint', 'contains', 'variable'])

tensor([  6,  11,  13,  22,  23,  25,  26,  36,  63,  64,  65,  72,  76,  82,
        113, 120, 121, 135, 137, 140, 144, 146, 149, 170, 172, 174, 179, 188,
        193, 196, 206, 207, 208, 220, 234, 241, 242, 243, 251, 254, 263, 264,
        270, 274, 286, 294, 300, 301, 307, 311, 321, 322, 327, 328, 336, 345,
        350, 356, 372, 373, 374, 375, 377, 379, 381, 382, 388, 394, 398, 399,
        400])
tensor([  6,  11,  13,  22,  23,  25,  26,  36,  63,  64,  65,  72,  76,  82,
        113, 120, 121, 135, 137, 140, 144, 146, 149, 170, 172, 174, 179, 188,
        193, 196, 206, 207, 208, 220, 234, 241, 242, 243, 251, 254, 263, 264,
        270, 274, 286, 294, 300, 301, 307, 311, 321, 322, 327, 328, 336, 345,
        350, 356, 372, 373, 374, 375, 377, 379, 381, 382, 388, 394, 398, 399,
        400])
tensor([2.6248e+02, 7.1923e+01, 3.6004e+02, 6.9008e+02, 4.4575e+01, 3.0870e+02,
        5.0718e+02, 2.8813e+02, 2.7882e+02, 4.9324e+02, 7.4327e+01, 3.8473e+02,
        6.2379e-02, 1.2578e+02, 

In [ ]:
batch_list = []
for batch in train_loader:
    batch_list.append(batch)
    if len(batch_list)>1:
      break

In [ ]:
batch_list[0]

BipartiteNodeDataBatch(constraint_features=[6917, 5], edge_index=[2, 140269], edge_attr=[140269, 1], variable_features=[12980, 19], candidates=[2145], nb_candidates=[32], candidate_choices=[32], candidate_scores=[2145], num_nodes=19897, batch=[19897], ptr=[33])

In [ ]:
batch_list[0].candidates

tensor([   32,    44,    45,  ..., 12941, 12949, 12971])

### Part 4: GNN Models





In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
DEVICE

device(type='cuda')

Next, I will define Gasse et al.'s graph neural network architecture.

In [ ]:
class GNNPolicy(torch.nn.Module):
    def __init__(self):
        super().__init__()
        emb_size = 64
        cons_nfeats = 5
        edge_nfeats = 1
        var_nfeats = 19

        # CONSTRAINT EMBEDDING
        self.cons_embedding = torch.nn.Sequential(
            torch.nn.LayerNorm(cons_nfeats),
            torch.nn.Linear(cons_nfeats, emb_size),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, emb_size),
            torch.nn.ReLU(),
        )

        # EDGE EMBEDDING
        self.edge_embedding = torch.nn.Sequential(
            torch.nn.LayerNorm(edge_nfeats),
        )

        # VARIABLE EMBEDDING
        self.var_embedding = torch.nn.Sequential(
            torch.nn.LayerNorm(var_nfeats),
            torch.nn.Linear(var_nfeats, emb_size),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, emb_size),
            torch.nn.ReLU(),
        )

        self.conv_v_to_c = BipartiteGraphConvolution()
        self.conv_c_to_v = BipartiteGraphConvolution()

        self.output_module = torch.nn.Sequential(
            torch.nn.Linear(emb_size, emb_size),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, 1, bias=False),
        )

    def forward(
        self, constraint_features, edge_indices, edge_features, variable_features
    ):
        reversed_edge_indices = torch.stack([edge_indices[1], edge_indices[0]], dim=0)

        # First step: linear embedding layers to a common dimension (64)
        constraint_features = self.cons_embedding(constraint_features)
        edge_features = self.edge_embedding(edge_features)
        variable_features = self.var_embedding(variable_features)

        # Two half convolutions
        constraint_features = self.conv_v_to_c(
            variable_features, reversed_edge_indices, edge_features, constraint_features
        )
        variable_features = self.conv_c_to_v(
            constraint_features, edge_indices, edge_features, variable_features
        )

        # A final MLP on the variable features
        output = self.output_module(variable_features).squeeze(-1)
        return output


class BipartiteGraphConvolution(torch_geometric.nn.MessagePassing):
    """
    The bipartite graph convolution is already provided by pytorch geometric and we merely need
    to provide the exact form of the messages being passed.
    """

    def __init__(self):
        super().__init__("add")
        emb_size = 64

        self.feature_module_left = torch.nn.Sequential(
            torch.nn.Linear(emb_size, emb_size)
        )
        self.feature_module_edge = torch.nn.Sequential(
            torch.nn.Linear(1, emb_size, bias=False)
        )
        self.feature_module_right = torch.nn.Sequential(
            torch.nn.Linear(emb_size, emb_size, bias=False)
        )
        self.feature_module_final = torch.nn.Sequential(
            torch.nn.LayerNorm(emb_size),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, emb_size),
        )

        self.post_conv_module = torch.nn.Sequential(torch.nn.LayerNorm(emb_size))

        # output_layers
        self.output_module = torch.nn.Sequential(
            torch.nn.Linear(2 * emb_size, emb_size),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, emb_size),
        )

    def forward(self, left_features, edge_indices, edge_features, right_features):
        """
        This method sends the messages, computed in the message method.
        """
        output = self.propagate(
            edge_indices,
            size=(left_features.shape[0], right_features.shape[0]),
            node_features=(left_features, right_features),
            edge_features=edge_features,
        )
        return self.output_module(
            torch.cat([self.post_conv_module(output), right_features], dim=-1)
        )

    def message(self, node_features_i, node_features_j, edge_features):
        output = self.feature_module_final(
            self.feature_module_left(node_features_i)
            + self.feature_module_edge(edge_features)
            + self.feature_module_right(node_features_j)
        )
        return output


policy = GNNPolicy().to(DEVICE)

With this model we can predict a probability distribution over actions as follows.

In [ ]:
observation = train_data[0].to(DEVICE)

logits = policy(
    observation.constraint_features,
    observation.edge_index,
    observation.edge_attr,
    observation.variable_features,
)
action_distribution = F.softmax(logits[observation.candidates], dim=-1)

print(action_distribution)

tensor([0.0140, 0.0139, 0.0141, 0.0142, 0.0141, 0.0141, 0.0141, 0.0141, 0.0142,
        0.0140, 0.0142, 0.0141, 0.0142, 0.0142, 0.0141, 0.0141, 0.0142, 0.0141,
        0.0139, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0142, 0.0139, 0.0142,
        0.0141, 0.0142, 0.0141, 0.0142, 0.0142, 0.0140, 0.0142, 0.0141, 0.0141,
        0.0141, 0.0139, 0.0141, 0.0141, 0.0142, 0.0141, 0.0142, 0.0141, 0.0139,
        0.0142, 0.0140, 0.0141, 0.0141, 0.0140, 0.0141, 0.0142, 0.0141, 0.0142,
        0.0141, 0.0141, 0.0139, 0.0141, 0.0141, 0.0142, 0.0139, 0.0141, 0.0140,
        0.0140, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0139, 0.0142],
       device='cuda:0', grad_fn=<SoftmaxBackward0>)


As can be seen, with randomly initialized weights, the initial distributions tend to be close to uniform.

Next, I will define my graph neural network architecture.

With this model we can predict a probability distribution over actions as follows.

As can be seen, with randomly initialized weights, the initial distributions tend to be close to uniform.

In [ ]:
class HeteroGATPolicyAlternative3(torch.nn.Module):
    def __init__(self, num_heads):
        super().__init__()
        emb_size = 64
        cons_nfeats = 5
        edge_nfeats = 1
        var_nfeats = 19

        # CONSTRAINT EMBEDDING
        self.cons_embedding = torch.nn.Sequential(
            torch.nn.LayerNorm(cons_nfeats),
            torch.nn.Linear(cons_nfeats, emb_size),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, emb_size),
            torch.nn.ReLU(),
        )

        # EDGE EMBEDDING
        self.edge_embedding = torch.nn.Sequential(
            torch.nn.LayerNorm(edge_nfeats),
        )

        # VARIABLE EMBEDDING
        self.var_embedding = torch.nn.Sequential(
            torch.nn.LayerNorm(var_nfeats),
            torch.nn.Linear(var_nfeats, emb_size),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, emb_size),
            torch.nn.ReLU(),
        )

        self.conv1 = HeteroConv({
                ('variable', 'participates', 'constraint'): GATConv((-1, -1), emb_size, heads=num_heads, add_self_loops=False),
                ('constraint', 'contains', 'variable'): GATConv((-1, -1), emb_size, heads=num_heads, add_self_loops=False),
            }, aggr='sum')

        self.lin1 = torch.nn.Linear(emb_size, emb_size)


        self.conv2 = HeteroConv({
                ('constraint', 'contains', 'variable'): GATConv((-1, -1),  (num_heads+1) * emb_size, heads=num_heads, add_self_loops=False, concat=False),
            }, aggr='sum')

        self.lin2 = torch.nn.Linear((num_heads+1) * emb_size, emb_size)


        self.output_module = torch.nn.Sequential(
            torch.nn.Linear((num_heads+2) * emb_size, emb_size),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, 1, bias=False),
        )

    def forward(self, x_dict, edge_index_dict, edge_attr_dict):

        x_dict['constraint'] = self.cons_embedding(x_dict['constraint'])
        x_dict['variable'] = self.var_embedding(x_dict['variable'])
        edge_attr_dict['variable', 'participates', 'constraint'] = self.edge_embedding(edge_attr_dict['variable', 'participates', 'constraint'])
        edge_attr_dict['constraint', 'contains', 'variable'] = self.edge_embedding(edge_attr_dict['constraint', 'contains', 'variable'])


        #Convolution Layer
        constraint_initial = x_dict['constraint']
        variable_initial = x_dict['variable']
        x_dict = self.conv1(x_dict, edge_index_dict, edge_attr_dict)
        x_dict['constraint'] = torch.cat([x_dict['constraint'], self.lin1(constraint_initial)], dim=-1)
        x_dict['variable'] = torch.cat([x_dict['variable'], self.lin1(variable_initial)], dim=-1)
        x_dict['constraint'] = x_dict['constraint'].relu()
        x_dict['variable'] = x_dict['variable'].relu()

        #Convolution Layer 2
        variable_second = x_dict['variable']
        x_dict = self.conv2(x_dict, edge_index_dict, edge_attr_dict)
        x_dict['variable'] = torch.cat([x_dict['variable'], self.lin2(variable_second)], dim=-1)

        #A final MLP on the variable features
        output = self.output_module(x_dict['variable']).squeeze(-1)

        return output

model_alt_3 = HeteroGATPolicyAlternative3(num_heads=1).to(DEVICE)
model_mult_head_4x4_alt_3 = HeteroGATPolicyAlternative3(num_heads=4).to(DEVICE)

/usr/local/lib/python3.10/site-packages/torch_geometric/nn/conv/hetero_conv.py:76: UserWarning: There exist node types ({'constraint'}) whose representations do not get updated during message passing as they do not occur as destination type in any edge type. This may lead to unexpected behavior.
  warnings.warn(


In [ ]:
class HeteroGATPolicyAlternative4(torch.nn.Module):
    def __init__(self, num_heads):
        super().__init__()
        emb_size = 64
        cons_nfeats = 5
        edge_nfeats = 1
        var_nfeats = 19

        # CONSTRAINT EMBEDDING
        self.cons_embedding = torch.nn.Sequential(
            torch.nn.LayerNorm(cons_nfeats),
            torch.nn.Linear(cons_nfeats, emb_size),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, emb_size),
            torch.nn.ReLU(),
        )

        # EDGE EMBEDDING
        self.edge_embedding = torch.nn.Sequential(
            torch.nn.LayerNorm(edge_nfeats),
        )

        # VARIABLE EMBEDDING
        self.var_embedding = torch.nn.Sequential(
            torch.nn.LayerNorm(var_nfeats),
            torch.nn.Linear(var_nfeats, emb_size),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, emb_size),
            torch.nn.ReLU(),
        )

        self.conv1 = HeteroConv({
                ('variable', 'participates', 'constraint'): GATConv((-1, -1), emb_size, heads=num_heads, add_self_loops=False),
                ('constraint', 'contains', 'variable'): GATConv((-1, -1), emb_size, heads=num_heads, add_self_loops=False),
            }, aggr='sum')

        self.lin1 = torch.nn.Linear(emb_size, emb_size)


        self.conv2 = HeteroConv({
                ('constraint', 'contains', 'variable'): GATConv((-1, -1),  (num_heads+1) * emb_size, heads=2, add_self_loops=False, concat=False),
            }, aggr='sum')

        self.lin2 = torch.nn.Linear((num_heads+1) * emb_size, emb_size)


        self.output_module = torch.nn.Sequential(
            torch.nn.Linear((num_heads+2) * emb_size, emb_size),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, 1, bias=False),
        )

    def forward(self, x_dict, edge_index_dict, edge_attr_dict):

        x_dict['constraint'] = self.cons_embedding(x_dict['constraint'])
        x_dict['variable'] = self.var_embedding(x_dict['variable'])
        edge_attr_dict['variable', 'participates', 'constraint'] = self.edge_embedding(edge_attr_dict['variable', 'participates', 'constraint'])
        edge_attr_dict['constraint', 'contains', 'variable'] = self.edge_embedding(edge_attr_dict['constraint', 'contains', 'variable'])


        #Convolution Layer
        constraint_initial = x_dict['constraint']
        variable_initial = x_dict['variable']
        x_dict = self.conv1(x_dict, edge_index_dict, edge_attr_dict)
        x_dict['constraint'] = torch.cat([x_dict['constraint'], self.lin1(constraint_initial)], dim=-1)
        x_dict['variable'] = torch.cat([x_dict['variable'], self.lin1(variable_initial)], dim=-1)
        x_dict['constraint'] = x_dict['constraint'].relu()
        x_dict['variable'] = x_dict['variable'].relu()

        #Convolution Layer 2
        variable_second = x_dict['variable']
        x_dict = self.conv2(x_dict, edge_index_dict, edge_attr_dict)
        x_dict['variable'] = torch.cat([x_dict['variable'], self.lin2(variable_second)], dim=-1)

        #A final MLP on the variable features
        output = self.output_module(x_dict['variable']).squeeze(-1)

        return output

model_mult_head_2_alt_3 = HeteroGATPolicyAlternative4(num_heads=2).to(DEVICE)
model_mult_head_4_alt_3 = HeteroGATPolicyAlternative4(num_heads=4).to(DEVICE)
model_multi_head_8_alt_3 = HeteroGATPolicyAlternative4(num_heads=8).to(DEVICE)
model_multi_head_8_alt_3_2 = HeteroGATPolicyAlternative4(num_heads=8).to(DEVICE)

In [ ]:
hetero_observation_alt_3 = hetero_train_data[0].to(DEVICE)

hetero_logits_alt_3 = model_alt_3(hetero_observation_alt_3.x_dict, hetero_observation_alt_3.edge_index_dict, hetero_observation_alt_3.edge_attr_dict)

action_distribution_gat_alt_3 = F.softmax(hetero_logits_alt_3[hetero_observation_alt_3.candidates], dim=-1)

print(action_distribution_gat_alt_3)

tensor([0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141],
       device='cuda:0', grad_fn=<SoftmaxBackward0>)


In [ ]:
hetero_observation_alt_3_2 = hetero_train_data[0].to(DEVICE)

hetero_logits_alt_3_2 = model_mult_head_2_alt_3(hetero_observation_alt_3_2.x_dict, hetero_observation_alt_3_2.edge_index_dict, hetero_observation_alt_3_2.edge_attr_dict)

action_distribution_gat_alt_3_2 = F.softmax(hetero_logits_alt_3_2[hetero_observation_alt_3_2.candidates], dim=-1)

print(action_distribution_gat_alt_3_2)

tensor([0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0142, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141],
       device='cuda:0', grad_fn=<SoftmaxBackward0>)


In [ ]:
hetero_observation_alt_3_4 = hetero_train_data[0].to(DEVICE)

hetero_logits_alt_3_4 = model_mult_head_4_alt_3(hetero_observation_alt_3_4.x_dict, hetero_observation_alt_3_4.edge_index_dict, hetero_observation_alt_3_4.edge_attr_dict)

action_distribution_gat_alt_3_4 = F.softmax(hetero_logits_alt_3_4[hetero_observation_alt_3_4.candidates], dim=-1)

print(action_distribution_gat_alt_3_4)

tensor([0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0140, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0140, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141],
       device='cuda:0', grad_fn=<SoftmaxBackward0>)


In [ ]:
hetero_observation_alt_3_4x4 = hetero_train_data[0].to(DEVICE)

hetero_logits_alt_3_4x4 = model_mult_head_4x4_alt_3(hetero_observation_alt_3_4x4.x_dict, hetero_observation_alt_3_4x4.edge_index_dict, hetero_observation_alt_3_4x4.edge_attr_dict)

action_distribution_gat_alt_3_4x4 = F.softmax(hetero_logits_alt_3_4x4[hetero_observation_alt_3_4x4.candidates], dim=-1)

print(action_distribution_gat_alt_3_4x4)


tensor([0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141],
       device='cuda:0', grad_fn=<SoftmaxBackward0>)


In [ ]:
hetero_observation_alt_3_8 = hetero_train_data[0].to(DEVICE)

hetero_logits_alt_3_8 = model_multi_head_8_alt_3(hetero_observation_alt_3_8.x_dict, hetero_observation_alt_3_8.edge_index_dict, hetero_observation_alt_3_8.edge_attr_dict)

action_distribution_gat_alt_3_8 = F.softmax(hetero_logits_alt_3_8[hetero_observation_alt_3_8.candidates], dim=-1)

print(action_distribution_gat_alt_3_8)

tensor([0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141,
        0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141, 0.0141],
       device='cuda:0', grad_fn=<SoftmaxBackward0>)


### Part 5: Training and Validation

Next, we will define two helper functions: one to train or evaluate the model on a whole epoch and compute metrics for monitoring, and one for padding tensors when doing predictions on a batch of graphs of potentially different number of variables.

In [ ]:
def process(policy, data_loader, optimizer=None):
    """
    This function will process a whole epoch of training or validation, depending on whether an optimizer is provided.
    """
    mean_loss = 0
    mean_acc = 0

    n_samples_processed = 0
    with torch.set_grad_enabled(optimizer is not None):
        for batch in data_loader:
            batch = batch.to(DEVICE)
            # Compute the logits (i.e. pre-softmax activations) according to the policy on the concatenated graphs
            logits = policy(
                batch.constraint_features,
                batch.edge_index,
                batch.edge_attr,
                batch.variable_features,
            )
            # Index the results by the candidates, and split and pad them
            logits = pad_tensor(logits[batch.candidates], batch.nb_candidates)
            # Compute the usual cross-entropy classification loss
            loss = F.cross_entropy(logits, batch.candidate_choices)

            if optimizer is not None:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            true_scores = pad_tensor(batch.candidate_scores, batch.nb_candidates)
            true_bestscore = true_scores.max(dim=-1, keepdims=True).values

            predicted_bestindex = logits.max(dim=-1, keepdims=True).indices
            accuracy = (
                (true_scores.gather(-1, predicted_bestindex) == true_bestscore)
                .float()
                .mean()
                .item()
            )

            mean_loss += loss.item() * batch.num_graphs
            mean_acc += accuracy * batch.num_graphs
            n_samples_processed += batch.num_graphs

    mean_loss /= n_samples_processed
    mean_acc /= n_samples_processed
    return mean_loss, mean_acc


def pad_tensor(input_, pad_sizes, pad_value=-1e8):
    """
    This utility function splits a tensor and pads each split to make them all the same size, then stacks them.
    """
    max_pad_size = pad_sizes.max()
    output = input_.split(pad_sizes.cpu().numpy().tolist())
    output = torch.stack(
        [
            F.pad(slice_, (0, max_pad_size - slice_.size(0)), "constant", pad_value)
            for slice_ in output
        ],
        dim=0,
    )
    return output

In [ ]:
def gat_process_batch(model, data_loader, optimizer=None):
    """
    This function will process a whole epoch of training or validation, depending on whether an optimizer is provided.
    """
    mean_loss = 0
    mean_acc = 0

    n_samples_processed = 0
    with torch.set_grad_enabled(optimizer is not None):
        for batch in data_loader:
            batch = batch.to(DEVICE)
            # Compute the logits (i.e. pre-softmax activations) according to the policy on the concatenated graphs
            logits = model(
                batch.x_dict,
                batch.edge_index_dict,
                batch.edge_attr_dict
            )
            # Index the results by the candidates, and split and pad them
            logits = pad_tensor(logits[batch.candidates], batch.nb_candidates)
            # Compute the usual cross-entropy classification loss
            loss = F.cross_entropy(logits, batch.candidate_choices)

            if optimizer is not None:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            true_scores = pad_tensor(batch.candidate_scores, batch.nb_candidates)
            true_bestscore = true_scores.max(dim=-1, keepdims=True).values

            predicted_bestindex = logits.max(dim=-1, keepdims=True).indices
            accuracy = (
                (true_scores.gather(-1, predicted_bestindex) == true_bestscore)
                .float()
                .mean()
                .item()
            )

            mean_loss += loss.item() * batch.num_graphs
            mean_acc += accuracy * batch.num_graphs
            n_samples_processed += batch.num_graphs

    mean_loss /= n_samples_processed
    mean_acc /= n_samples_processed
    return mean_loss, mean_acc


def pad_tensor(input_, pad_sizes, pad_value=-1e8):
    """
    This utility function splits a tensor and pads each split to make them all the same size, then stacks them.
    """
    max_pad_size = pad_sizes.max()
    output = input_.split(pad_sizes.cpu().numpy().tolist())
    output = torch.stack(
        [
            F.pad(slice_, (0, max_pad_size - slice_.size(0)), "constant", pad_value)
            for slice_ in output
        ],
        dim=0,
    )
    return output

In [ ]:
def my_gat_process_batch(model, data_loader, optimizer=None):
    """
    This function will process a whole epoch of training or validation, depending on whether an optimizer is provided.
    """
    mean_loss = 0
    mean_acc = 0

    n_samples_processed = 0
    with torch.set_grad_enabled(optimizer is not None):
        for batch in data_loader:
            batch = batch.to(DEVICE)
            # Compute the logits (i.e. pre-softmax activations) according to the policy on the concatenated graphs
            logits = model(
                batch.x_dict,
                batch.edge_index_dict,
                batch.edge_attr_dict
            )
            # Index the results by the candidates, and split and pad them
            logits = pad_tensor(logits[batch.candidates], batch.nb_candidates)
            # Compute the usual cross-entropy classification loss
            loss = F.binary_cross_entropy(F.softmax(logits, dim=-1), my_utility_func(batch.candidate_choices, batch.nb_candidates).cuda())

            if optimizer is not None:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            true_scores = pad_tensor(batch.candidate_scores, batch.nb_candidates)
            true_bestscore = true_scores.max(dim=-1, keepdims=True).values

            predicted_bestindex = logits.max(dim=-1, keepdims=True).indices
            accuracy = (
                (true_scores.gather(-1, predicted_bestindex) == true_bestscore)
                .float()
                .mean()
                .item()
            )

            mean_loss += loss.item() * batch.num_graphs
            mean_acc += accuracy * batch.num_graphs
            n_samples_processed += batch.num_graphs

    mean_loss /= n_samples_processed
    mean_acc /= n_samples_processed
    return mean_loss, mean_acc


def pad_tensor(input_, pad_sizes, pad_value=-1e8):
    """
    This utility function splits a tensor and pads each split to make them all the same size, then stacks them.
    """
    max_pad_size = pad_sizes.max()
    output = input_.split(pad_sizes.cpu().numpy().tolist())
    output = torch.stack(
        [
            F.pad(slice_, (0, max_pad_size - slice_.size(0)), "constant", pad_value)
            for slice_ in output
        ],
        dim=0,
    )
    return output


def my_utility_func(candidate_choices, nb_candidates):
  x = torch.zeros((len(nb_candidates),max(nb_candidates)))
  for i,item in enumerate(candidate_choices):
    x[i,item] = 1
  return x

After this, we can actually create the model and train it: GCNN


In [ ]:
optimizer = torch.optim.Adam(policy.parameters(), lr=LEARNING_RATE)
for epoch in range(NB_EPOCHS):
    print(f"Epoch {epoch+1}")

    train_loss, train_acc = process(policy, train_loader, optimizer)
    print(f"Train loss: {train_loss:0.3f}, accuracy {train_acc:0.3f}")

    valid_loss, valid_acc = process(policy, valid_loader, None)
    print(f"Valid loss: {valid_loss:0.3f}, accuracy {valid_acc:0.3f}")

torch.save(policy.state_dict(), "drive/MyDrive/Thesis/ca_trained_params.pkl")

Epoch 1
Train loss: 2.789, accuracy 0.507
Valid loss: 2.780, accuracy 0.484
Epoch 2
Train loss: 2.658, accuracy 0.527
Valid loss: 2.736, accuracy 0.523
Epoch 3
Train loss: 2.618, accuracy 0.535
Valid loss: 2.609, accuracy 0.545
Epoch 4
Train loss: 2.593, accuracy 0.540
Valid loss: 2.586, accuracy 0.547
Epoch 5
Train loss: 2.578, accuracy 0.543
Valid loss: 2.566, accuracy 0.549
Epoch 6
Train loss: 2.561, accuracy 0.544
Valid loss: 2.555, accuracy 0.541
Epoch 7
Train loss: 2.553, accuracy 0.547
Valid loss: 2.545, accuracy 0.566
Epoch 8
Train loss: 2.544, accuracy 0.549
Valid loss: 2.569, accuracy 0.560
Epoch 9
Train loss: 2.540, accuracy 0.549
Valid loss: 2.550, accuracy 0.547
Epoch 10
Train loss: 2.533, accuracy 0.550
Valid loss: 2.535, accuracy 0.558
Epoch 11
Train loss: 2.525, accuracy 0.552
Valid loss: 2.558, accuracy 0.538
Epoch 12
Train loss: 2.521, accuracy 0.551
Valid loss: 2.520, accuracy 0.548
Epoch 13
Train loss: 2.516, accuracy 0.552
Valid loss: 2.555, accuracy 0.544
Epoch 14

After this, we can actually create the model and train it: GAT with 1 Attention Head


After this, we can actually create the model and train it: GAT with 2 Attention Heads


After this, we can actually create the model and train it: GAT with 4 Attention Heads



After this, we can actually create the model and train it: GAT with 8 Attention Heads

In [ ]:
gat_optimizer_batch = torch.optim.Adam(model_alt_3.parameters(), lr=LEARNING_RATE)
for epoch in range(NB_EPOCHS):
    print(f"Epoch {epoch+1}")

    gat_train_loss_batch, gat_train_acc_batch = gat_process_batch(model_alt_3, hetero_train_loader, gat_optimizer_batch)
    print(f"Train loss: {gat_train_loss_batch:0.3f}, accuracy {gat_train_acc_batch:0.3f}")

    gat_valid_loss_batch, gat_valid_acc_batch = gat_process_batch(model_alt_3, hetero_valid_loader, None)
    print(f"Valid loss: {gat_valid_loss_batch:0.3f}, accuracy {gat_valid_acc_batch:0.3f}")

torch.save(model_alt_3.state_dict(), "drive/MyDrive/Thesis/ca_gat_trained_params_batch.pkl")

Epoch 1
Train loss: 2.804, accuracy 0.502
Valid loss: 2.675, accuracy 0.531
Epoch 2
Train loss: 2.661, accuracy 0.527
Valid loss: 2.650, accuracy 0.506
Epoch 3
Train loss: 2.626, accuracy 0.534
Valid loss: 2.634, accuracy 0.538
Epoch 4
Train loss: 2.606, accuracy 0.540
Valid loss: 2.589, accuracy 0.528
Epoch 5
Train loss: 2.588, accuracy 0.541
Valid loss: 2.571, accuracy 0.542
Epoch 6
Train loss: 2.576, accuracy 0.544
Valid loss: 2.568, accuracy 0.536
Epoch 7
Train loss: 2.564, accuracy 0.545
Valid loss: 2.568, accuracy 0.541
Epoch 8
Train loss: 2.554, accuracy 0.545
Valid loss: 2.594, accuracy 0.552
Epoch 9
Train loss: 2.546, accuracy 0.547
Valid loss: 2.563, accuracy 0.538
Epoch 10
Train loss: 2.542, accuracy 0.548
Valid loss: 2.537, accuracy 0.544
Epoch 11
Train loss: 2.535, accuracy 0.550
Valid loss: 2.574, accuracy 0.517
Epoch 12
Train loss: 2.529, accuracy 0.548
Valid loss: 2.529, accuracy 0.535
Epoch 13
Train loss: 2.524, accuracy 0.550
Valid loss: 2.533, accuracy 0.538
Epoch 14

In [ ]:
mh_gat_optimizer_2_alt_3_batch = torch.optim.Adam(model_mult_head_2_alt_3.parameters(), lr=LEARNING_RATE)
for epoch in range(NB_EPOCHS):
    print(f"Epoch {epoch+1}")

    mh_gat_train_loss_2_alt_3_batch, mh_gat_train_acc_2_alt_3_batch = gat_process_batch(model_mult_head_2_alt_3, hetero_train_loader, mh_gat_optimizer_2_alt_3_batch)
    print(f"Train loss: {mh_gat_train_loss_2_alt_3_batch:0.3f}, accuracy {mh_gat_train_acc_2_alt_3_batch:0.3f}")

    mh_gat_valid_loss_2_alt_3_batch, mh_gat_valid_acc_2_alt_3_batch = gat_process_batch(model_mult_head_2_alt_3, hetero_valid_loader, None)
    print(f"Valid loss: {mh_gat_valid_loss_2_alt_3_batch:0.3f}, accuracy {mh_gat_valid_acc_2_alt_3_batch:0.3f}")

torch.save(model_mult_head_2_alt_3.state_dict(), "drive/MyDrive/Thesis/ca_mh_gat_trained_params_2_alt_3_batch.pkl")

Epoch 1
Train loss: 2.787, accuracy 0.517
Valid loss: 2.625, accuracy 0.517
Epoch 2
Train loss: 2.617, accuracy 0.534
Valid loss: 2.601, accuracy 0.533
Epoch 3
Train loss: 2.579, accuracy 0.541
Valid loss: 2.555, accuracy 0.554
Epoch 4
Train loss: 2.556, accuracy 0.543
Valid loss: 2.539, accuracy 0.548
Epoch 5
Train loss: 2.543, accuracy 0.546
Valid loss: 2.560, accuracy 0.563
Epoch 6
Train loss: 2.531, accuracy 0.548
Valid loss: 2.524, accuracy 0.546
Epoch 7
Train loss: 2.520, accuracy 0.549
Valid loss: 2.524, accuracy 0.546
Epoch 8
Train loss: 2.508, accuracy 0.550
Valid loss: 2.544, accuracy 0.532
Epoch 9
Train loss: 2.498, accuracy 0.553
Valid loss: 2.505, accuracy 0.553
Epoch 10
Train loss: 2.492, accuracy 0.552
Valid loss: 2.489, accuracy 0.563
Epoch 11
Train loss: 2.485, accuracy 0.555
Valid loss: 2.474, accuracy 0.559
Epoch 12
Train loss: 2.480, accuracy 0.555
Valid loss: 2.473, accuracy 0.554
Epoch 13
Train loss: 2.473, accuracy 0.559
Valid loss: 2.486, accuracy 0.555
Epoch 14

In [ ]:
mh_gat_optimizer_4_alt_3_batch = torch.optim.Adam(model_mult_head_4_alt_3.parameters(), lr=LEARNING_RATE)
for epoch in range(NB_EPOCHS):
    print(f"Epoch {epoch+1}")

    mh_gat_train_loss_4_alt_3_batch, mh_gat_train_acc_4_alt_3_batch = gat_process_batch(model_mult_head_4_alt_3, hetero_train_loader, mh_gat_optimizer_4_alt_3_batch)
    print(f"Train loss: {mh_gat_train_loss_4_alt_3_batch:0.3f}, accuracy {mh_gat_train_acc_4_alt_3_batch:0.3f}")

    mh_gat_valid_loss_4_alt_3_batch, mh_gat_valid_acc_4_alt_3_batch = gat_process_batch(model_mult_head_4_alt_3, hetero_valid_loader, None)
    print(f"Valid loss: {mh_gat_valid_loss_4_alt_3_batch:0.3f}, accuracy {mh_gat_valid_acc_4_alt_3_batch:0.3f}")

torch.save(model_mult_head_4_alt_3.state_dict(), "drive/MyDrive/Thesis/ca_mh_gat_trained_params_4_alt_3_batch.pkl")

Epoch 1
Train loss: 2.876, accuracy 0.517
Valid loss: 2.758, accuracy 0.525
Epoch 2
Train loss: 2.701, accuracy 0.540
Valid loss: 2.655, accuracy 0.536
Epoch 3
Train loss: 2.651, accuracy 0.547
Valid loss: 2.635, accuracy 0.539
Epoch 4
Train loss: 2.629, accuracy 0.548
Valid loss: 2.623, accuracy 0.559
Epoch 5
Train loss: 2.609, accuracy 0.554
Valid loss: 2.588, accuracy 0.543
Epoch 6
Train loss: 2.598, accuracy 0.554
Valid loss: 2.615, accuracy 0.546
Epoch 7
Train loss: 2.592, accuracy 0.554
Valid loss: 2.569, accuracy 0.556
Epoch 8
Train loss: 2.582, accuracy 0.556
Valid loss: 2.585, accuracy 0.542
Epoch 9
Train loss: 2.579, accuracy 0.555
Valid loss: 2.636, accuracy 0.512
Epoch 10
Train loss: 2.574, accuracy 0.556
Valid loss: 2.566, accuracy 0.547
Epoch 11
Train loss: 2.570, accuracy 0.553
Valid loss: 2.578, accuracy 0.539
Epoch 12
Train loss: 2.566, accuracy 0.557
Valid loss: 2.560, accuracy 0.553
Epoch 13
Train loss: 2.560, accuracy 0.557
Valid loss: 2.578, accuracy 0.546
Epoch 14

In [ ]:
mh_gat_optimizer_4x4_alt_3_batch = torch.optim.Adam(model_mult_head_4x4_alt_3.parameters(), lr=LEARNING_RATE)
for epoch in range(NB_EPOCHS):
    print(f"Epoch {epoch+1}")

    mh_gat_train_loss_4x4_alt_3_batch, mh_gat_train_acc_4x4_alt_3_batch = gat_process_batch(model_mult_head_4x4_alt_3, hetero_train_loader, mh_gat_optimizer_4x4_alt_3_batch)
    print(f"Train loss: {mh_gat_train_loss_4x4_alt_3_batch:0.3f}, accuracy {mh_gat_train_acc_4x4_alt_3_batch:0.3f}")

    mh_gat_valid_loss_4x4_alt_3_batch, mh_gat_valid_acc_4x4_alt_3_batch = gat_process_batch(model_mult_head_4x4_alt_3, hetero_valid_loader, None)
    print(f"Valid loss: {mh_gat_valid_loss_4x4_alt_3_batch:0.3f}, accuracy {mh_gat_valid_acc_4x4_alt_3_batch:0.3f}")

torch.save(model_mult_head_4x4_alt_3.state_dict(), "drive/MyDrive/Thesis/ca_mh_gat_trained_params_4x4_alt_3_batch.pkl")

Epoch 1
Train loss: 2.821, accuracy 0.521
Valid loss: 2.672, accuracy 0.542
Epoch 2
Train loss: 2.653, accuracy 0.538
Valid loss: 2.651, accuracy 0.541
Epoch 3
Train loss: 2.592, accuracy 0.540
Valid loss: 2.576, accuracy 0.516
Epoch 4
Train loss: 2.564, accuracy 0.542
Valid loss: 2.590, accuracy 0.520
Epoch 5
Train loss: 2.546, accuracy 0.544
Valid loss: 2.556, accuracy 0.548
Epoch 6
Train loss: 2.533, accuracy 0.547
Valid loss: 2.540, accuracy 0.545
Epoch 7
Train loss: 2.520, accuracy 0.549
Valid loss: 2.518, accuracy 0.539
Epoch 8
Train loss: 2.506, accuracy 0.551
Valid loss: 2.520, accuracy 0.550
Epoch 9
Train loss: 2.497, accuracy 0.554
Valid loss: 2.494, accuracy 0.568
Epoch 10
Train loss: 2.488, accuracy 0.556
Valid loss: 2.498, accuracy 0.548
Epoch 11
Train loss: 2.480, accuracy 0.557
Valid loss: 2.488, accuracy 0.554
Epoch 12
Train loss: 2.475, accuracy 0.556
Valid loss: 2.503, accuracy 0.548
Epoch 13
Train loss: 2.471, accuracy 0.558
Valid loss: 2.479, accuracy 0.542
Epoch 14

In [ ]:
mh_gat_optimizer_8_alt_3_batch = torch.optim.Adam(model_multi_head_8_alt_3.parameters(), lr=LEARNING_RATE)
for epoch in range(NB_EPOCHS):
    print(f"Epoch {epoch+1}")

    mh_gat_train_loss_8_alt_3_batch, mh_gat_train_acc_8_alt_3_batch = gat_process_batch(model_multi_head_8_alt_3, hetero_train_loader, mh_gat_optimizer_8_alt_3_batch)
    print(f"Train loss: {mh_gat_train_loss_4_alt_3_batch:0.3f}, accuracy {mh_gat_train_acc_4_alt_3_batch:0.3f}")

    mh_gat_valid_loss_8_alt_3_batch, mh_gat_valid_acc_8_alt_3_batch = gat_process_batch(model_multi_head_8_alt_3, hetero_valid_loader, None)
    print(f"Valid loss: {mh_gat_valid_loss_8_alt_3_batch:0.3f}, accuracy {mh_gat_valid_acc_8_alt_3_batch:0.3f}")

torch.save(model_multi_head_8_alt_3.state_dict(), "drive/MyDrive/Thesis/ca_mh_gat_trained_params_8_alt_3_batch.pkl")

Epoch 1
Train loss: 2.492, accuracy 0.565
Valid loss: 2.711, accuracy 0.540
Epoch 2
Train loss: 2.492, accuracy 0.565
Valid loss: 2.640, accuracy 0.554
Epoch 3
Train loss: 2.492, accuracy 0.565
Valid loss: 2.609, accuracy 0.554
Epoch 4
Train loss: 2.492, accuracy 0.565
Valid loss: 2.602, accuracy 0.551
Epoch 5
Train loss: 2.492, accuracy 0.565
Valid loss: 2.582, accuracy 0.557
Epoch 6
Train loss: 2.492, accuracy 0.565
Valid loss: 2.571, accuracy 0.548
Epoch 7
Train loss: 2.492, accuracy 0.565
Valid loss: 2.580, accuracy 0.549
Epoch 8
Train loss: 2.492, accuracy 0.565
Valid loss: 2.567, accuracy 0.563
Epoch 9
Train loss: 2.492, accuracy 0.565
Valid loss: 2.557, accuracy 0.558
Epoch 10
Train loss: 2.492, accuracy 0.565
Valid loss: 2.561, accuracy 0.561
Epoch 11
Train loss: 2.492, accuracy 0.565
Valid loss: 2.559, accuracy 0.549
Epoch 12
Train loss: 2.492, accuracy 0.565
Valid loss: 2.576, accuracy 0.546
Epoch 13
Train loss: 2.492, accuracy 0.565
Valid loss: 2.560, accuracy 0.558
Epoch 14

In [ ]:
mh_gat_optimizer_8_alt_3_batch_2 = torch.optim.Adam(model_multi_head_8_alt_3_2.parameters(), lr=LEARNING_RATE)
for epoch in range(NB_EPOCHS):
    print(f"Epoch {epoch+1}")

    mh_gat_train_loss_8_alt_3_batch_2, mh_gat_train_acc_8_alt_3_batch_2 = my_gat_process_batch(model_multi_head_8_alt_3_2, hetero_train_loader, mh_gat_optimizer_8_alt_3_batch_2)
    print(f"Train loss: {mh_gat_train_loss_8_alt_3_batch_2:0.3f}, accuracy {mh_gat_train_acc_8_alt_3_batch_2:0.3f}")

    mh_gat_valid_loss_8_alt_3_batch_2, mh_gat_valid_acc_8_alt_3_batch_2 = my_gat_process_batch(model_multi_head_8_alt_3_2, hetero_valid_loader, None)
    print(f"Valid loss: {mh_gat_valid_loss_8_alt_3_batch_2:0.3f}, accuracy {mh_gat_valid_acc_8_alt_3_batch_2:0.3f}")

torch.save(model_multi_head_8_alt_3_2.state_dict(), "drive/MyDrive/Thesis/ca_mh_gat_trained_params_8_alt_3_batch_2.pkl")

Epoch 1
Train loss: 0.041, accuracy 0.522
Valid loss: 0.039, accuracy 0.522
Epoch 2
Train loss: 0.039, accuracy 0.541
Valid loss: 0.038, accuracy 0.556
Epoch 3
Train loss: 0.038, accuracy 0.540
Valid loss: 0.038, accuracy 0.525
Epoch 4
Train loss: 0.038, accuracy 0.539
Valid loss: 0.038, accuracy 0.561
Epoch 5
Train loss: 0.037, accuracy 0.542
Valid loss: 0.037, accuracy 0.547
Epoch 6
Train loss: 0.037, accuracy 0.545
Valid loss: 0.037, accuracy 0.526
Epoch 7
Train loss: 0.037, accuracy 0.549
Valid loss: 0.037, accuracy 0.554
Epoch 8
Train loss: 0.037, accuracy 0.548
Valid loss: 0.037, accuracy 0.562
Epoch 9
Train loss: 0.037, accuracy 0.551
Valid loss: 0.037, accuracy 0.550
Epoch 10
Train loss: 0.037, accuracy 0.552
Valid loss: 0.037, accuracy 0.569
Epoch 11
Train loss: 0.037, accuracy 0.554
Valid loss: 0.037, accuracy 0.552
Epoch 12
Train loss: 0.037, accuracy 0.555
Valid loss: 0.037, accuracy 0.550
Epoch 13
Train loss: 0.037, accuracy 0.557
Valid loss: 0.036, accuracy 0.552
Epoch 14

### Part 6: Evaluation

Finally, we can evaluate the performance of the model. We first define appropriate environments. For benchmarking purposes, we include a trivial environment that merely runs SCIP.

In [ ]:
policy.load_state_dict(torch.load("drive/MyDrive/Thesis/ca_trained_params.pkl", map_location=DEVICE, weights_only=True))
policy.eval()

GNNPolicy(
  (cons_embedding): Sequential(
    (0): LayerNorm((5,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=5, out_features=64, bias=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=64, bias=True)
    (4): ReLU()
  )
  (edge_embedding): Sequential(
    (0): LayerNorm((1,), eps=1e-05, elementwise_affine=True)
  )
  (var_embedding): Sequential(
    (0): LayerNorm((19,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=19, out_features=64, bias=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=64, bias=True)
    (4): ReLU()
  )
  (conv_v_to_c): BipartiteGraphConvolution()
  (conv_c_to_v): BipartiteGraphConvolution()
  (output_module): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=False)
  )
)

In [ ]:
model_alt_3.load_state_dict(torch.load("drive/MyDrive/Thesis/ca_gat_trained_params_batch.pkl", map_location=DEVICE, weights_only=True))
model_alt_3.eval()

HeteroGATPolicyAlternative3(
  (cons_embedding): Sequential(
    (0): LayerNorm((5,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=5, out_features=64, bias=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=64, bias=True)
    (4): ReLU()
  )
  (edge_embedding): Sequential(
    (0): LayerNorm((1,), eps=1e-05, elementwise_affine=True)
  )
  (var_embedding): Sequential(
    (0): LayerNorm((19,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=19, out_features=64, bias=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=64, bias=True)
    (4): ReLU()
  )
  (conv1): HeteroConv(num_relations=2)
  (lin1): Linear(in_features=64, out_features=64, bias=True)
  (conv2): HeteroConv(num_relations=1)
  (lin2): Linear(in_features=128, out_features=64, bias=True)
  (output_module): Sequential(
    (0): Linear(in_features=192, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=False)
  )
)

In [ ]:
model_mult_head_2_alt_3.load_state_dict(torch.load("drive/MyDrive/Thesis/ca_mh_gat_trained_params_2_alt_3_batch.pkl", map_location=DEVICE, weights_only=True))
model_mult_head_2_alt_3.eval()

HeteroGATPolicyAlternative4(
  (cons_embedding): Sequential(
    (0): LayerNorm((5,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=5, out_features=64, bias=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=64, bias=True)
    (4): ReLU()
  )
  (edge_embedding): Sequential(
    (0): LayerNorm((1,), eps=1e-05, elementwise_affine=True)
  )
  (var_embedding): Sequential(
    (0): LayerNorm((19,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=19, out_features=64, bias=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=64, bias=True)
    (4): ReLU()
  )
  (conv1): HeteroConv(num_relations=2)
  (lin1): Linear(in_features=64, out_features=64, bias=True)
  (conv2): HeteroConv(num_relations=1)
  (lin2): Linear(in_features=192, out_features=64, bias=True)
  (output_module): Sequential(
    (0): Linear(in_features=256, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=False)
  )
)

In [ ]:
model_mult_head_4_alt_3.load_state_dict(torch.load("drive/MyDrive/Thesis/ca_mh_gat_trained_params_4_alt_3_batch.pkl", map_location=DEVICE, weights_only=True))
model_mult_head_4_alt_3.eval()

HeteroGATPolicyAlternative4(
  (cons_embedding): Sequential(
    (0): LayerNorm((5,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=5, out_features=64, bias=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=64, bias=True)
    (4): ReLU()
  )
  (edge_embedding): Sequential(
    (0): LayerNorm((1,), eps=1e-05, elementwise_affine=True)
  )
  (var_embedding): Sequential(
    (0): LayerNorm((19,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=19, out_features=64, bias=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=64, bias=True)
    (4): ReLU()
  )
  (conv1): HeteroConv(num_relations=2)
  (lin1): Linear(in_features=64, out_features=64, bias=True)
  (conv2): HeteroConv(num_relations=1)
  (lin2): Linear(in_features=320, out_features=64, bias=True)
  (output_module): Sequential(
    (0): Linear(in_features=384, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=False)
  )
)

In [ ]:
model_multi_head_8_alt_3.load_state_dict(torch.load("drive/MyDrive/Thesis/ca_mh_gat_trained_params_8_alt_3_batch.pkl", map_location=DEVICE, weights_only=True))
model_multi_head_8_alt_3.eval()

HeteroGATPolicyAlternative4(
  (cons_embedding): Sequential(
    (0): LayerNorm((5,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=5, out_features=64, bias=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=64, bias=True)
    (4): ReLU()
  )
  (edge_embedding): Sequential(
    (0): LayerNorm((1,), eps=1e-05, elementwise_affine=True)
  )
  (var_embedding): Sequential(
    (0): LayerNorm((19,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=19, out_features=64, bias=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=64, bias=True)
    (4): ReLU()
  )
  (conv1): HeteroConv(num_relations=2)
  (lin1): Linear(in_features=64, out_features=64, bias=True)
  (conv2): HeteroConv(num_relations=1)
  (lin2): Linear(in_features=576, out_features=64, bias=True)
  (output_module): Sequential(
    (0): Linear(in_features=640, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=False)
  )
)

In [ ]:
model_multi_head_8_alt_3_2.load_state_dict(torch.load("drive/MyDrive/Thesis/ca_mh_gat_trained_params_8_alt_3_batch_2.pkl", map_location=DEVICE, weights_only=True))
model_multi_head_8_alt_3_2.eval()

HeteroGATPolicyAlternative4(
  (cons_embedding): Sequential(
    (0): LayerNorm((5,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=5, out_features=64, bias=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=64, bias=True)
    (4): ReLU()
  )
  (edge_embedding): Sequential(
    (0): LayerNorm((1,), eps=1e-05, elementwise_affine=True)
  )
  (var_embedding): Sequential(
    (0): LayerNorm((19,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=19, out_features=64, bias=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=64, bias=True)
    (4): ReLU()
  )
  (conv1): HeteroConv(num_relations=2)
  (lin1): Linear(in_features=64, out_features=64, bias=True)
  (conv2): HeteroConv(num_relations=1)
  (lin2): Linear(in_features=576, out_features=64, bias=True)
  (output_module): Sequential(
    (0): Linear(in_features=640, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=False)
  )
)

In [ ]:
# Print optimizer's state_dict
print("Optimizer's state_dict:")
for var_name in model_multi_head_8_alt_3_2.state_dict():
    print(var_name, "\t", model_multi_head_8_alt_3_2.state_dict()[var_name])

Optimizer's state_dict:
cons_embedding.0.weight 	 tensor([0.6689, 0.7092, 0.1972, 1.4590, 1.4025], device='cuda:0')
cons_embedding.0.bias 	 tensor([-0.0934,  0.5143, -0.3650,  0.1733,  0.0377], device='cuda:0')
cons_embedding.1.weight 	 tensor([[-0.1544, -0.0762, -0.0505, -0.0866,  0.1412],
        [-0.1312,  0.2768, -0.1490,  0.2835,  0.0427],
        [ 0.1710,  0.0193,  0.2399,  0.7954,  0.0317],
        [ 0.0472,  0.1163,  0.1629,  0.2957,  0.0854],
        [ 0.0137, -0.2371,  0.0505,  0.8070, -1.0729],
        [ 0.3014, -0.0424,  0.1961, -0.0366,  0.1172],
        [ 0.1245,  0.3179,  0.1754, -0.0275,  0.0600],
        [-0.3589, -0.5929, -0.1016,  0.2140,  0.3305],
        [ 0.1607,  0.7164, -0.4953,  0.0548, -0.3322],
        [-0.0580,  0.2741, -0.1733, -0.7866,  0.4233],
        [-0.0079,  0.2687,  0.0359,  0.3059,  0.0399],
        [ 0.1259,  0.1944,  0.4786, -0.9700,  0.4306],
        [-0.2050,  0.1192,  0.0756,  0.0507,  0.0854],
        [ 0.4015,  0.6312, -0.5067, -0.3873, -0.

In [ ]:
scip_parameters = {
    "separating/maxrounds": 0,
    "presolving/maxrestarts": 0,
    "limits/time": 3600,
}
env = ecole.environment.Branching(
    observation_function=ecole.observation.NodeBipartite(),
    information_function={
        "nb_nodes": ecole.reward.NNodes(),
        "time": ecole.reward.SolvingTime(),
    },
    scip_params=scip_parameters,
)
gat_env = ecole.environment.Branching(
    observation_function=ecole.observation.NodeBipartite(),
    information_function={
        "nb_nodes": ecole.reward.NNodes(),
        "time": ecole.reward.SolvingTime(),
    },
    scip_params=scip_parameters,
)
mh_gat_env_b = ecole.environment.Branching(
    observation_function=ecole.observation.NodeBipartite(),
    information_function={
        "nb_nodes": ecole.reward.NNodes(),
        "time": ecole.reward.SolvingTime(),
    },
    scip_params=scip_parameters,
)
mh_gat_env = ecole.environment.Branching(
    observation_function=ecole.observation.NodeBipartite(),
    information_function={
        "nb_nodes": ecole.reward.NNodes(),
        "time": ecole.reward.SolvingTime(),
    },
    scip_params=scip_parameters,
)
mh_gat_env_2 = ecole.environment.Branching(
    observation_function=ecole.observation.NodeBipartite(),
    information_function={
        "nb_nodes": ecole.reward.NNodes(),
        "time": ecole.reward.SolvingTime(),
    },
    scip_params=scip_parameters,
)
mh_gat_env_4 = ecole.environment.Branching(
    observation_function=ecole.observation.NodeBipartite(),
    information_function={
        "nb_nodes": ecole.reward.NNodes(),
        "time": ecole.reward.SolvingTime(),
    },
    scip_params=scip_parameters,
)
default_env = ecole.environment.Configuring(
    observation_function=None,
    information_function={
        "nb_nodes": ecole.reward.NNodes(),
        "time": ecole.reward.SolvingTime(),
    },
    scip_params=scip_parameters,
)

scip_parameters_2 = {
    "separating/maxrounds": 0,
    "presolving/maxrestarts": 0,
    "limits/time": 3600,
    "branching/allfullstrong/priority": 10000,
    "branching/relpscost/priority": -10000,
}

scip_parameters_3 = {
    "separating/maxrounds": 0,
    "presolving/maxrestarts": 0,
    "limits/time": 3600,
    "branching/mostinf/priority": 10000,
    "branching/allfullstrong/priority": -10000,
    "branching/relpscost/priority": -10000,
}

fsb_env = ecole.environment.Configuring(
    observation_function=None,
    information_function={
        "nb_nodes": ecole.reward.NNodes(),
        "time": ecole.reward.SolvingTime(),
    },
    scip_params=scip_parameters_2,
)

mib_env = ecole.environment.Configuring(
    observation_function=None,
    information_function={
        "nb_nodes": ecole.reward.NNodes(),
        "time": ecole.reward.SolvingTime(),
    },
    scip_params=scip_parameters_3,
)

In [ ]:
instances = ecole.instance.CombinatorialAuctionGenerator(n_items=100, n_bids=500, min_value=1, max_value=100,
                       value_deviation=0.5, add_item_prob=0.65, max_n_sub_bids=5,
                       additivity=0.2, budget_factor=1.5, resale_factor=0.5,
                       integers=False, warnings=False)
for instance_count, instance in zip(range(NB_EVAL_INSTANCES), instances):
    # Run the GNN brancher
    nb_nodes, time = 0, 0
    observation, action_set, _, done, info = env.reset(instance)
    nb_nodes += info["nb_nodes"]
    time += info["time"]
    while not done:
        with torch.no_grad():
            observation = (
                torch.from_numpy(observation.row_features.astype(np.float32)).to(DEVICE),
                torch.from_numpy(observation.edge_features.indices.astype(np.int64)).to(DEVICE),
                torch.from_numpy(observation.edge_features.values.astype(np.float32)).view(-1, 1).to(DEVICE),
                torch.from_numpy(observation.variable_features.astype(np.float32)).to(DEVICE),
            )
            logits = policy(*observation)
            action = action_set[logits[action_set.astype(np.int64)].argmax()]
            observation, action_set, _, done, info = env.step(action)
        nb_nodes += info["nb_nodes"]
        time += info["time"]

    # Run the GAT brancher
    #gat_nb_nodes, gat_time = 0, 0
    #gat_observation, gat_action_set, _, gat_done, gat_info = gat_env.reset(instance)
    #gat_nb_nodes += gat_info["nb_nodes"]
    #gat_time += gat_info["time"]
    #while not gat_done:
    #    with torch.no_grad():
    #        graph = HeteroData()
    #        #node features
    #        graph['constraint'].x = torch.FloatTensor(gat_observation.row_features).to(DEVICE)
    #        graph['variable'].x = torch.FloatTensor(gat_observation.variable_features).to(DEVICE)
    #        #edge indices
    #        graph['variable', 'participates', 'constraint'].edge_index = torch.stack([torch.LongTensor(gat_observation.edge_features.indices.astype(np.int32)[1]), torch.LongTensor(gat_observation.edge_features.indices.astype(np.int32)[0])], dim=0).to(DEVICE)
    #        graph['constraint', 'contains', 'variable'].edge_index = torch.LongTensor(gat_observation.edge_features.indices.astype(np.int32)).to(DEVICE)
    #        #edge features
    #        graph['variable', 'participates', 'constraint'].edge_attr = torch.FloatTensor(np.expand_dims(gat_observation.edge_features.values, axis=-1)).to(DEVICE)
    #        graph['constraint', 'contains', 'variable'].edge_attr = torch.FloatTensor(np.expand_dims(gat_observation.edge_features.values, axis=-1)).to(DEVICE)
 #
    #       gat_logits = model_alt_3(graph.x_dict, graph.edge_index_dict, graph.edge_attr_dict)
    #        gat_action = gat_action_set[gat_logits[gat_action_set.astype(np.int64)].argmax()]
    #        gat_observation, gat_action_set, _, gat_done, gat_info = gat_env.step(gat_action)
    #    gat_nb_nodes += gat_info["nb_nodes"]
    #    gat_time += gat_info["time"]

    # Run the Multi Head (2) GAT brancher
    #mh_gat_nb_nodes_2, mh_gat_time_2 = 0, 0
    #mh_gat_observation_2, mh_gat_action_set_2, _, mh_gat_done_2, mh_gat_info_2 = mh_gat_env_2.reset(instance)
    #mh_gat_nb_nodes_2 += mh_gat_info_2["nb_nodes"]
    #mh_gat_time_2 += mh_gat_info_2["time"]
    #while not mh_gat_done_2:
    #    with torch.no_grad():
    #        graph = HeteroData()
    #        #node features
    #        graph['constraint'].x = torch.FloatTensor(mh_gat_observation_2.row_features).to(DEVICE)
    #        graph['variable'].x = torch.FloatTensor(mh_gat_observation_2.variable_features).to(DEVICE)
    #        #edge indices
    #        graph['variable', 'participates', 'constraint'].edge_index = torch.stack([torch.LongTensor(mh_gat_observation_2.edge_features.indices.astype(np.int32)[1]), torch.LongTensor(mh_gat_observation_2.edge_features.indices.astype(np.int32)[0])], dim=0).to(DEVICE)
    #        graph['constraint', 'contains', 'variable'].edge_index = torch.LongTensor(mh_gat_observation_2.edge_features.indices.astype(np.int32)).to(DEVICE)
    #        #edge features
    #        graph['variable', 'participates', 'constraint'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation_2.edge_features.values, axis=-1)).to(DEVICE)
    #        graph['constraint', 'contains', 'variable'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation_2.edge_features.values, axis=-1)).to(DEVICE)

    #        mh_gat_logits_2 = model_mult_head_2_alt_3(graph.x_dict, graph.edge_index_dict, graph.edge_attr_dict)
    #        mh_gat_action_2 = mh_gat_action_set_2[mh_gat_logits_2[mh_gat_action_set_2.astype(np.int64)].argmax()]
    #        mh_gat_observation_2, mh_gat_action_set_2, _, mh_gat_done_2, mh_gat_info_2 = mh_gat_env_2.step(mh_gat_action_2)
    #    mh_gat_nb_nodes_2 += mh_gat_info_2["nb_nodes"]
    #    mh_gat_time_2 += mh_gat_info_2["time"]

    # Run the Multi Head (4) GAT brancher
    #mh_gat_nb_nodes_4, mh_gat_time_4 = 0, 0
    #mh_gat_observation_4, mh_gat_action_set_4, _, mh_gat_done_4, mh_gat_info_4 = mh_gat_env_4.reset(instance)
    #mh_gat_nb_nodes_4 += mh_gat_info_4["nb_nodes"]
    #mh_gat_time_4 += mh_gat_info_4["time"]
    #while not mh_gat_done_4:
    #    with torch.no_grad():
    #        graph = HeteroData()
    #        #node features
    #        graph['constraint'].x = torch.FloatTensor(mh_gat_observation_4.row_features).to(DEVICE)
    #        graph['variable'].x = torch.FloatTensor(mh_gat_observation_4.variable_features).to(DEVICE)
    #        #edge indices
    #        graph['variable', 'participates', 'constraint'].edge_index = torch.stack([torch.LongTensor(mh_gat_observation_4.edge_features.indices.astype(np.int32)[1]), torch.LongTensor(mh_gat_observation_4.edge_features.indices.astype(np.int32)[0])], dim=0).to(DEVICE)
    #        graph['constraint', 'contains', 'variable'].edge_index = torch.LongTensor(mh_gat_observation_4.edge_features.indices.astype(np.int32)).to(DEVICE)
    #        #edge features
    #        graph['variable', 'participates', 'constraint'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation_4.edge_features.values, axis=-1)).to(DEVICE)
    #        graph['constraint', 'contains', 'variable'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation_4.edge_features.values, axis=-1)).to(DEVICE)

    #       mh_gat_logits_4 = model_mult_head_4x4_alt_3(graph.x_dict, graph.edge_index_dict, graph.edge_attr_dict)
    #        mh_gat_action_4 = mh_gat_action_set_4[mh_gat_logits_4[mh_gat_action_set_4.astype(np.int64)].argmax()]
    #        mh_gat_observation_4, mh_gat_action_set_4, _, mh_gat_done_4, mh_gat_info_4 = mh_gat_env_4.step(mh_gat_action_4)
    #    mh_gat_nb_nodes_4 += mh_gat_info_4["nb_nodes"]
    #    mh_gat_time_4 += mh_gat_info_4["time"]

    # Run the Multi Head (8) GAT brancher
    mh_gat_nb_nodes, mh_gat_time = 0, 0
    mh_gat_observation, mh_gat_action_set, _, mh_gat_done, mh_gat_info = mh_gat_env.reset(instance)
    mh_gat_nb_nodes += mh_gat_info["nb_nodes"]
    mh_gat_time += mh_gat_info["time"]
    while not mh_gat_done:
        with torch.no_grad():
            graph = HeteroData()
            #node features
            graph['constraint'].x = torch.FloatTensor(mh_gat_observation.row_features).to(DEVICE)
            graph['variable'].x = torch.FloatTensor(mh_gat_observation.variable_features).to(DEVICE)
            #edge indices
            graph['variable', 'participates', 'constraint'].edge_index = torch.stack([torch.LongTensor(mh_gat_observation.edge_features.indices.astype(np.int32)[1]), torch.LongTensor(mh_gat_observation.edge_features.indices.astype(np.int32)[0])], dim=0).to(DEVICE)
            graph['constraint', 'contains', 'variable'].edge_index = torch.LongTensor(mh_gat_observation.edge_features.indices.astype(np.int32)).to(DEVICE)
            #edge features
            graph['variable', 'participates', 'constraint'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation.edge_features.values, axis=-1)).to(DEVICE)
            graph['constraint', 'contains', 'variable'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation.edge_features.values, axis=-1)).to(DEVICE)

            mh_gat_logits = model_multi_head_8_alt_3(graph.x_dict, graph.edge_index_dict, graph.edge_attr_dict)
            mh_gat_action = mh_gat_action_set[mh_gat_logits[mh_gat_action_set.astype(np.int64)].argmax()]
            mh_gat_observation, mh_gat_action_set, _, mh_gat_done, mh_gat_info = mh_gat_env.step(mh_gat_action)
        mh_gat_nb_nodes += mh_gat_info["nb_nodes"]
        mh_gat_time += mh_gat_info["time"]

    # Run the Multi Head (8) GAT 2 brancher
    mh_gat_nb_nodes_b, mh_gat_time_b = 0, 0
    mh_gat_observation_b, mh_gat_action_set_b, _, mh_gat_done_b, mh_gat_info_b = mh_gat_env_b.reset(instance)
    mh_gat_nb_nodes_b += mh_gat_info_b["nb_nodes"]
    mh_gat_time_b += mh_gat_info_b["time"]
    while not mh_gat_done_b:
        with torch.no_grad():
            graph = HeteroData()
            #node features
            graph['constraint'].x = torch.FloatTensor(mh_gat_observation_b.row_features).to(DEVICE)
            graph['variable'].x = torch.FloatTensor(mh_gat_observation_b.variable_features).to(DEVICE)
            #edge indices
            graph['variable', 'participates', 'constraint'].edge_index = torch.stack([torch.LongTensor(mh_gat_observation_b.edge_features.indices.astype(np.int32)[1]), torch.LongTensor(mh_gat_observation_b.edge_features.indices.astype(np.int32)[0])], dim=0).to(DEVICE)
            graph['constraint', 'contains', 'variable'].edge_index = torch.LongTensor(mh_gat_observation_b.edge_features.indices.astype(np.int32)).to(DEVICE)
            #edge features
            graph['variable', 'participates', 'constraint'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation_b.edge_features.values, axis=-1)).to(DEVICE)
            graph['constraint', 'contains', 'variable'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation_b.edge_features.values, axis=-1)).to(DEVICE)

            mh_gat_logits_b = model_multi_head_8_alt_3_2(graph.x_dict, graph.edge_index_dict, graph.edge_attr_dict)
            mh_gat_action_b = mh_gat_action_set_b[mh_gat_logits_b[mh_gat_action_set_b.astype(np.int64)].argmax()]
            mh_gat_observation_b, mh_gat_action_set_b, _, mh_gat_done_b, mh_gat_info_b = mh_gat_env_b.step(mh_gat_action_b)
        mh_gat_nb_nodes_b += mh_gat_info_b["nb_nodes"]
        mh_gat_time_b += mh_gat_info_b["time"]


    # Run SCIP's default brancher
    default_env.reset(instance)
    _, _, _, _, default_info = default_env.step({})

    # Run Strong Branching
    fsb_env.reset(instance)
    _, _, _, _, fsb_info = fsb_env.step({})

    # Run Most Infeasible Branching
    mib_env.reset(instance)
    _, _, _, _, mib_info = mib_env.step({})

    print(f"Instance {instance_count: >3} | FSB nb nodes    {int(fsb_info['nb_nodes']): >4d}  | FSB time   {fsb_info['time']: >6.2f} ")
    print(f"Instance {instance_count: >3} | MIB nb nodes    {int(mib_info['nb_nodes']): >4d}  | MIB time   {mib_info['time']: >6.2f} ")
    print(f"Instance {instance_count: >3} | SCIP nb nodes    {int(default_info['nb_nodes']): >4d}  | SCIP time   {default_info['time']: >6.2f} ")
    print(f"             | GNN  nb nodes    {int(nb_nodes): >4d}  | GNN  time   {time: >6.2f} ")
    #print(f"             | GAT  nb nodes    {int(gat_nb_nodes): >4d}  | GAT  time   {gat_time: >6.2f} ")
    #print(f"             | MH GAT(2) nb nodes {int(mh_gat_nb_nodes_2): >4d}  | MH GAT(2) time   {mh_gat_time_2: >6.2f} ")
    #print(f"             | MH GAT(4) nb nodes {int(mh_gat_nb_nodes_4): >4d}  | MH GAT(4) time   {mh_gat_time_4: >6.2f} ")
    print(f"             | MH GAT(8) nb nodes {int(mh_gat_nb_nodes): >4d}  | MH GAT(8) time   {mh_gat_time: >6.2f} ")
    print(f"             | MH GAT(8) nb nodes {int(mh_gat_nb_nodes_b): >4d}  | MH GAT(8) time   {mh_gat_time_b: >6.2f} ")

Instance   0 | FSB nb nodes       3  | FSB time    15.34 
Instance   0 | MIB nb nodes     277  | MIB time     2.08 
Instance   0 | SCIP nb nodes      15  | SCIP time     2.79 
             | GNN  nb nodes      51  | GNN  time     1.41 
             | MH GAT(8) nb nodes   71  | MH GAT(8) time     1.53 
             | MH GAT(8) nb nodes   49  | MH GAT(8) time     1.43 
Instance   1 | FSB nb nodes       1  | FSB time     3.38 
Instance   1 | MIB nb nodes      93  | MIB time     1.47 
Instance   1 | SCIP nb nodes       3  | SCIP time     1.70 
             | GNN  nb nodes      29  | GNN  time     1.18 
             | MH GAT(8) nb nodes   21  | MH GAT(8) time     1.12 
             | MH GAT(8) nb nodes   27  | MH GAT(8) time     1.15 
Instance   2 | FSB nb nodes       5  | FSB time     4.07 
Instance   2 | MIB nb nodes     123  | MIB time     1.13 
Instance   2 | SCIP nb nodes       3  | SCIP time     1.15 
             | GNN  nb nodes      31  | GNN  time     0.97 
             | MH GAT(8)

In [ ]:
instances = ecole.instance.CombinatorialAuctionGenerator(n_items=200, n_bids=1000, min_value=1, max_value=100,
                       value_deviation=0.5, add_item_prob=0.65, max_n_sub_bids=5,
                       additivity=0.2, budget_factor=1.5, resale_factor=0.5,
                       integers=False, warnings=False)
for instance_count, instance in zip(range(NB_EVAL_INSTANCES), instances):
    # Run the GNN brancher
    nb_nodes, time = 0, 0
    observation, action_set, _, done, info = env.reset(instance)
    nb_nodes += info["nb_nodes"]
    time += info["time"]
    while not done:
        with torch.no_grad():
            observation = (
                torch.from_numpy(observation.row_features.astype(np.float32)).to(DEVICE),
                torch.from_numpy(observation.edge_features.indices.astype(np.int64)).to(DEVICE),
                torch.from_numpy(observation.edge_features.values.astype(np.float32)).view(-1, 1).to(DEVICE),
                torch.from_numpy(observation.variable_features.astype(np.float32)).to(DEVICE),
            )
            logits = policy(*observation)
            action = action_set[logits[action_set.astype(np.int64)].argmax()]
            observation, action_set, _, done, info = env.step(action)
        nb_nodes += info["nb_nodes"]
        time += info["time"]

    # Run the GAT brancher
    #gat_nb_nodes, gat_time = 0, 0
    #gat_observation, gat_action_set, _, gat_done, gat_info = gat_env.reset(instance)
    #gat_nb_nodes += gat_info["nb_nodes"]
    #gat_time += gat_info["time"]
    #while not gat_done:
    #    with torch.no_grad():
    #        graph = HeteroData()
    #        #node features
    #        graph['constraint'].x = torch.FloatTensor(gat_observation.row_features).to(DEVICE)
    #        graph['variable'].x = torch.FloatTensor(gat_observation.variable_features).to(DEVICE)
    #        #edge indices
    #        graph['variable', 'participates', 'constraint'].edge_index = torch.stack([torch.LongTensor(gat_observation.edge_features.indices.astype(np.int32)[1]), torch.LongTensor(gat_observation.edge_features.indices.astype(np.int32)[0])], dim=0).to(DEVICE)
    #        graph['constraint', 'contains', 'variable'].edge_index = torch.LongTensor(gat_observation.edge_features.indices.astype(np.int32)).to(DEVICE)
    #        #edge features
    #        graph['variable', 'participates', 'constraint'].edge_attr = torch.FloatTensor(np.expand_dims(gat_observation.edge_features.values, axis=-1)).to(DEVICE)
    #        graph['constraint', 'contains', 'variable'].edge_attr = torch.FloatTensor(np.expand_dims(gat_observation.edge_features.values, axis=-1)).to(DEVICE)
 #
    #       gat_logits = model_alt_3(graph.x_dict, graph.edge_index_dict, graph.edge_attr_dict)
    #        gat_action = gat_action_set[gat_logits[gat_action_set.astype(np.int64)].argmax()]
    #        gat_observation, gat_action_set, _, gat_done, gat_info = gat_env.step(gat_action)
    #    gat_nb_nodes += gat_info["nb_nodes"]
    #    gat_time += gat_info["time"]

    # Run the Multi Head (2) GAT brancher
    #mh_gat_nb_nodes_2, mh_gat_time_2 = 0, 0
    #mh_gat_observation_2, mh_gat_action_set_2, _, mh_gat_done_2, mh_gat_info_2 = mh_gat_env_2.reset(instance)
    #mh_gat_nb_nodes_2 += mh_gat_info_2["nb_nodes"]
    #mh_gat_time_2 += mh_gat_info_2["time"]
    #while not mh_gat_done_2:
    #    with torch.no_grad():
    #        graph = HeteroData()
    #        #node features
    #        graph['constraint'].x = torch.FloatTensor(mh_gat_observation_2.row_features).to(DEVICE)
    #        graph['variable'].x = torch.FloatTensor(mh_gat_observation_2.variable_features).to(DEVICE)
    #        #edge indices
    #        graph['variable', 'participates', 'constraint'].edge_index = torch.stack([torch.LongTensor(mh_gat_observation_2.edge_features.indices.astype(np.int32)[1]), torch.LongTensor(mh_gat_observation_2.edge_features.indices.astype(np.int32)[0])], dim=0).to(DEVICE)
    #        graph['constraint', 'contains', 'variable'].edge_index = torch.LongTensor(mh_gat_observation_2.edge_features.indices.astype(np.int32)).to(DEVICE)
    #        #edge features
    #        graph['variable', 'participates', 'constraint'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation_2.edge_features.values, axis=-1)).to(DEVICE)
    #        graph['constraint', 'contains', 'variable'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation_2.edge_features.values, axis=-1)).to(DEVICE)

    #        mh_gat_logits_2 = model_mult_head_2_alt_3(graph.x_dict, graph.edge_index_dict, graph.edge_attr_dict)
    #        mh_gat_action_2 = mh_gat_action_set_2[mh_gat_logits_2[mh_gat_action_set_2.astype(np.int64)].argmax()]
    #        mh_gat_observation_2, mh_gat_action_set_2, _, mh_gat_done_2, mh_gat_info_2 = mh_gat_env_2.step(mh_gat_action_2)
    #    mh_gat_nb_nodes_2 += mh_gat_info_2["nb_nodes"]
    #    mh_gat_time_2 += mh_gat_info_2["time"]

    # Run the Multi Head (4) GAT brancher
    #mh_gat_nb_nodes_4, mh_gat_time_4 = 0, 0
    #mh_gat_observation_4, mh_gat_action_set_4, _, mh_gat_done_4, mh_gat_info_4 = mh_gat_env_4.reset(instance)
    #mh_gat_nb_nodes_4 += mh_gat_info_4["nb_nodes"]
    #mh_gat_time_4 += mh_gat_info_4["time"]
    #while not mh_gat_done_4:
    #    with torch.no_grad():
    #        graph = HeteroData()
    #        #node features
    #        graph['constraint'].x = torch.FloatTensor(mh_gat_observation_4.row_features).to(DEVICE)
    #        graph['variable'].x = torch.FloatTensor(mh_gat_observation_4.variable_features).to(DEVICE)
    #        #edge indices
    #        graph['variable', 'participates', 'constraint'].edge_index = torch.stack([torch.LongTensor(mh_gat_observation_4.edge_features.indices.astype(np.int32)[1]), torch.LongTensor(mh_gat_observation_4.edge_features.indices.astype(np.int32)[0])], dim=0).to(DEVICE)
    #        graph['constraint', 'contains', 'variable'].edge_index = torch.LongTensor(mh_gat_observation_4.edge_features.indices.astype(np.int32)).to(DEVICE)
    #        #edge features
    #        graph['variable', 'participates', 'constraint'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation_4.edge_features.values, axis=-1)).to(DEVICE)
    #        graph['constraint', 'contains', 'variable'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation_4.edge_features.values, axis=-1)).to(DEVICE)

    #       mh_gat_logits_4 = model_mult_head_4x4_alt_3(graph.x_dict, graph.edge_index_dict, graph.edge_attr_dict)
    #        mh_gat_action_4 = mh_gat_action_set_4[mh_gat_logits_4[mh_gat_action_set_4.astype(np.int64)].argmax()]
    #        mh_gat_observation_4, mh_gat_action_set_4, _, mh_gat_done_4, mh_gat_info_4 = mh_gat_env_4.step(mh_gat_action_4)
    #    mh_gat_nb_nodes_4 += mh_gat_info_4["nb_nodes"]
    #    mh_gat_time_4 += mh_gat_info_4["time"]

    # Run the Multi Head (8) GAT brancher
    #mh_gat_nb_nodes, mh_gat_time = 0, 0
    #mh_gat_observation, mh_gat_action_set, _, mh_gat_done, mh_gat_info = mh_gat_env.reset(instance)
    #mh_gat_nb_nodes += mh_gat_info["nb_nodes"]
    #mh_gat_time += mh_gat_info["time"]
    #while not mh_gat_done:
    #    with torch.no_grad():
    #        graph = HeteroData()
    #        #node features
    #        graph['constraint'].x = torch.FloatTensor(mh_gat_observation.row_features).to(DEVICE)
    #        graph['variable'].x = torch.FloatTensor(mh_gat_observation.variable_features).to(DEVICE)
    #        #edge indices
    #        graph['variable', 'participates', 'constraint'].edge_index = torch.stack([torch.LongTensor(mh_gat_observation.edge_features.indices.astype(np.int32)[1]), torch.LongTensor(mh_gat_observation.edge_features.indices.astype(np.int32)[0])], dim=0).to(DEVICE)
    #        graph['constraint', 'contains', 'variable'].edge_index = torch.LongTensor(mh_gat_observation.edge_features.indices.astype(np.int32)).to(DEVICE)
    #        #edge features
    #        graph['variable', 'participates', 'constraint'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation.edge_features.values, axis=-1)).to(DEVICE)
    #        graph['constraint', 'contains', 'variable'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation.edge_features.values, axis=-1)).to(DEVICE)

    #        mh_gat_logits = model_multi_head_8_alt_3(graph.x_dict, graph.edge_index_dict, graph.edge_attr_dict)
    #        mh_gat_action = mh_gat_action_set[mh_gat_logits[mh_gat_action_set.astype(np.int64)].argmax()]
    #        mh_gat_observation, mh_gat_action_set, _, mh_gat_done, mh_gat_info = mh_gat_env.step(mh_gat_action)
    #    mh_gat_nb_nodes += mh_gat_info["nb_nodes"]
    #    mh_gat_time += mh_gat_info["time"]

    # Run the Multi Head (8) GAT 2 brancher
    mh_gat_nb_nodes_b, mh_gat_time_b = 0, 0
    mh_gat_observation_b, mh_gat_action_set_b, _, mh_gat_done_b, mh_gat_info_b = mh_gat_env_b.reset(instance)
    mh_gat_nb_nodes_b += mh_gat_info_b["nb_nodes"]
    mh_gat_time_b += mh_gat_info_b["time"]
    while not mh_gat_done_b:
        with torch.no_grad():
            graph = HeteroData()
            #node features
            graph['constraint'].x = torch.FloatTensor(mh_gat_observation_b.row_features).to(DEVICE)
            graph['variable'].x = torch.FloatTensor(mh_gat_observation_b.variable_features).to(DEVICE)
            #edge indices
            graph['variable', 'participates', 'constraint'].edge_index = torch.stack([torch.LongTensor(mh_gat_observation_b.edge_features.indices.astype(np.int32)[1]), torch.LongTensor(mh_gat_observation_b.edge_features.indices.astype(np.int32)[0])], dim=0).to(DEVICE)
            graph['constraint', 'contains', 'variable'].edge_index = torch.LongTensor(mh_gat_observation_b.edge_features.indices.astype(np.int32)).to(DEVICE)
            #edge features
            graph['variable', 'participates', 'constraint'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation_b.edge_features.values, axis=-1)).to(DEVICE)
            graph['constraint', 'contains', 'variable'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation_b.edge_features.values, axis=-1)).to(DEVICE)

            mh_gat_logits_b = model_multi_head_8_alt_3_2(graph.x_dict, graph.edge_index_dict, graph.edge_attr_dict)
            mh_gat_action_b = mh_gat_action_set_b[mh_gat_logits_b[mh_gat_action_set_b.astype(np.int64)].argmax()]
            mh_gat_observation_b, mh_gat_action_set_b, _, mh_gat_done_b, mh_gat_info_b = mh_gat_env_b.step(mh_gat_action_b)
        mh_gat_nb_nodes_b += mh_gat_info_b["nb_nodes"]
        mh_gat_time_b += mh_gat_info_b["time"]


    # Run SCIP's default brancher
    default_env.reset(instance)
    _, _, _, _, default_info = default_env.step({})

    # Run Strong Branching
    #fsb_env.reset(instance)
    #_, _, _, _, fsb_info = fsb_env.step({})

    # Run Most Infeasible Branching
    #mib_env.reset(instance)
    #_, _, _, _, mib_info = mib_env.step({})

    #print(f"Instance {instance_count: >3} | FSB nb nodes    {int(fsb_info['nb_nodes']): >4d}  | FSB time   {fsb_info['time']: >6.2f} ")
    #print(f"Instance {instance_count: >3} | MIB nb nodes    {int(mib_info['nb_nodes']): >4d}  | MIB time   {mib_info['time']: >6.2f} ")
    print(f"Instance {instance_count: >3} | SCIP nb nodes    {int(default_info['nb_nodes']): >4d}  | SCIP time   {default_info['time']: >6.2f} ")
    print(f"             | GNN  nb nodes    {int(nb_nodes): >4d}  | GNN  time   {time: >6.2f} ")
    #print(f"             | GAT  nb nodes    {int(gat_nb_nodes): >4d}  | GAT  time   {gat_time: >6.2f} ")
    #print(f"             | MH GAT(2) nb nodes {int(mh_gat_nb_nodes_2): >4d}  | MH GAT(2) time   {mh_gat_time_2: >6.2f} ")
    #print(f"             | MH GAT(4) nb nodes {int(mh_gat_nb_nodes_4): >4d}  | MH GAT(4) time   {mh_gat_time_4: >6.2f} ")
    #print(f"             | MH GAT(8) nb nodes {int(mh_gat_nb_nodes): >4d}  | MH GAT(8) time   {mh_gat_time: >6.2f} ")
    print(f"             | MH GAT(8) nb nodes {int(mh_gat_nb_nodes_b): >4d}  | MH GAT(8) time   {mh_gat_time_b: >6.2f} ")

Instance   0 | SCIP nb nodes    2016  | SCIP time    22.28 
             | GNN  nb nodes    1981  | GNN  time    16.85 
             | MH GAT(8) nb nodes 2119  | MH GAT(8) time    21.40 
Instance   1 | SCIP nb nodes    2905  | SCIP time    29.26 
             | GNN  nb nodes    2649  | GNN  time    23.51 
             | MH GAT(8) nb nodes 3442  | MH GAT(8) time    32.56 
Instance   2 | SCIP nb nodes    1488  | SCIP time    19.75 
             | GNN  nb nodes    1411  | GNN  time    14.82 
             | MH GAT(8) nb nodes 1691  | MH GAT(8) time    17.02 
Instance   3 | SCIP nb nodes     638  | SCIP time    14.31 
             | GNN  nb nodes     551  | GNN  time     8.26 
             | MH GAT(8) nb nodes  501  | MH GAT(8) time     8.97 
Instance   4 | SCIP nb nodes      83  | SCIP time    10.41 
             | GNN  nb nodes     241  | GNN  time     5.85 
             | MH GAT(8) nb nodes  234  | MH GAT(8) time     5.91 
Instance   5 | SCIP nb nodes      49  | SCIP time     9.50 
     

In [ ]:
instances = ecole.instance.CombinatorialAuctionGenerator(n_items=300, n_bids=1500, min_value=1, max_value=100,
                       value_deviation=0.5, add_item_prob=0.65, max_n_sub_bids=5,
                       additivity=0.2, budget_factor=1.5, resale_factor=0.5,
                       integers=False, warnings=False)
for instance_count, instance in zip(range(NB_EVAL_INSTANCES), instances):
    # Run the GNN brancher
    nb_nodes, time = 0, 0
    observation, action_set, _, done, info = env.reset(instance)
    nb_nodes += info["nb_nodes"]
    time += info["time"]
    while not done:
        with torch.no_grad():
            observation = (
                torch.from_numpy(observation.row_features.astype(np.float32)).to(DEVICE),
                torch.from_numpy(observation.edge_features.indices.astype(np.int64)).to(DEVICE),
                torch.from_numpy(observation.edge_features.values.astype(np.float32)).view(-1, 1).to(DEVICE),
                torch.from_numpy(observation.variable_features.astype(np.float32)).to(DEVICE),
            )
            logits = policy(*observation)
            action = action_set[logits[action_set.astype(np.int64)].argmax()]
            observation, action_set, _, done, info = env.step(action)
        nb_nodes += info["nb_nodes"]
        time += info["time"]

    # Run the GAT brancher
    #gat_nb_nodes, gat_time = 0, 0
    #gat_observation, gat_action_set, _, gat_done, gat_info = gat_env.reset(instance)
    #gat_nb_nodes += gat_info["nb_nodes"]
    #gat_time += gat_info["time"]
    #while not gat_done:
    #    with torch.no_grad():
    #        graph = HeteroData()
    #        #node features
    #        graph['constraint'].x = torch.FloatTensor(gat_observation.row_features).to(DEVICE)
    #        graph['variable'].x = torch.FloatTensor(gat_observation.variable_features).to(DEVICE)
    #        #edge indices
    #        graph['variable', 'participates', 'constraint'].edge_index = torch.stack([torch.LongTensor(gat_observation.edge_features.indices.astype(np.int32)[1]), torch.LongTensor(gat_observation.edge_features.indices.astype(np.int32)[0])], dim=0).to(DEVICE)
    #        graph['constraint', 'contains', 'variable'].edge_index = torch.LongTensor(gat_observation.edge_features.indices.astype(np.int32)).to(DEVICE)
    #        #edge features
    #        graph['variable', 'participates', 'constraint'].edge_attr = torch.FloatTensor(np.expand_dims(gat_observation.edge_features.values, axis=-1)).to(DEVICE)
    #        graph['constraint', 'contains', 'variable'].edge_attr = torch.FloatTensor(np.expand_dims(gat_observation.edge_features.values, axis=-1)).to(DEVICE)
 #
    #       gat_logits = model_alt_3(graph.x_dict, graph.edge_index_dict, graph.edge_attr_dict)
    #        gat_action = gat_action_set[gat_logits[gat_action_set.astype(np.int64)].argmax()]
    #        gat_observation, gat_action_set, _, gat_done, gat_info = gat_env.step(gat_action)
    #    gat_nb_nodes += gat_info["nb_nodes"]
    #    gat_time += gat_info["time"]

    # Run the Multi Head (2) GAT brancher
    #mh_gat_nb_nodes_2, mh_gat_time_2 = 0, 0
    #mh_gat_observation_2, mh_gat_action_set_2, _, mh_gat_done_2, mh_gat_info_2 = mh_gat_env_2.reset(instance)
    #mh_gat_nb_nodes_2 += mh_gat_info_2["nb_nodes"]
    #mh_gat_time_2 += mh_gat_info_2["time"]
    #while not mh_gat_done_2:
    #    with torch.no_grad():
    #        graph = HeteroData()
    #        #node features
    #        graph['constraint'].x = torch.FloatTensor(mh_gat_observation_2.row_features).to(DEVICE)
    #        graph['variable'].x = torch.FloatTensor(mh_gat_observation_2.variable_features).to(DEVICE)
    #        #edge indices
    #        graph['variable', 'participates', 'constraint'].edge_index = torch.stack([torch.LongTensor(mh_gat_observation_2.edge_features.indices.astype(np.int32)[1]), torch.LongTensor(mh_gat_observation_2.edge_features.indices.astype(np.int32)[0])], dim=0).to(DEVICE)
    #        graph['constraint', 'contains', 'variable'].edge_index = torch.LongTensor(mh_gat_observation_2.edge_features.indices.astype(np.int32)).to(DEVICE)
    #        #edge features
    #        graph['variable', 'participates', 'constraint'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation_2.edge_features.values, axis=-1)).to(DEVICE)
    #        graph['constraint', 'contains', 'variable'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation_2.edge_features.values, axis=-1)).to(DEVICE)

    #        mh_gat_logits_2 = model_mult_head_2_alt_3(graph.x_dict, graph.edge_index_dict, graph.edge_attr_dict)
    #        mh_gat_action_2 = mh_gat_action_set_2[mh_gat_logits_2[mh_gat_action_set_2.astype(np.int64)].argmax()]
    #        mh_gat_observation_2, mh_gat_action_set_2, _, mh_gat_done_2, mh_gat_info_2 = mh_gat_env_2.step(mh_gat_action_2)
    #    mh_gat_nb_nodes_2 += mh_gat_info_2["nb_nodes"]
    #    mh_gat_time_2 += mh_gat_info_2["time"]

    # Run the Multi Head (4) GAT brancher
    #mh_gat_nb_nodes_4, mh_gat_time_4 = 0, 0
    #mh_gat_observation_4, mh_gat_action_set_4, _, mh_gat_done_4, mh_gat_info_4 = mh_gat_env_4.reset(instance)
    #mh_gat_nb_nodes_4 += mh_gat_info_4["nb_nodes"]
    #mh_gat_time_4 += mh_gat_info_4["time"]
    #while not mh_gat_done_4:
    #    with torch.no_grad():
    #        graph = HeteroData()
    #        #node features
    #        graph['constraint'].x = torch.FloatTensor(mh_gat_observation_4.row_features).to(DEVICE)
    #        graph['variable'].x = torch.FloatTensor(mh_gat_observation_4.variable_features).to(DEVICE)
    #        #edge indices
    #        graph['variable', 'participates', 'constraint'].edge_index = torch.stack([torch.LongTensor(mh_gat_observation_4.edge_features.indices.astype(np.int32)[1]), torch.LongTensor(mh_gat_observation_4.edge_features.indices.astype(np.int32)[0])], dim=0).to(DEVICE)
    #        graph['constraint', 'contains', 'variable'].edge_index = torch.LongTensor(mh_gat_observation_4.edge_features.indices.astype(np.int32)).to(DEVICE)
    #        #edge features
    #        graph['variable', 'participates', 'constraint'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation_4.edge_features.values, axis=-1)).to(DEVICE)
    #        graph['constraint', 'contains', 'variable'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation_4.edge_features.values, axis=-1)).to(DEVICE)

    #       mh_gat_logits_4 = model_mult_head_4x4_alt_3(graph.x_dict, graph.edge_index_dict, graph.edge_attr_dict)
    #        mh_gat_action_4 = mh_gat_action_set_4[mh_gat_logits_4[mh_gat_action_set_4.astype(np.int64)].argmax()]
    #        mh_gat_observation_4, mh_gat_action_set_4, _, mh_gat_done_4, mh_gat_info_4 = mh_gat_env_4.step(mh_gat_action_4)
    #    mh_gat_nb_nodes_4 += mh_gat_info_4["nb_nodes"]
    #    mh_gat_time_4 += mh_gat_info_4["time"]

    # Run the Multi Head (8) GAT brancher
    #mh_gat_nb_nodes, mh_gat_time = 0, 0
    #mh_gat_observation, mh_gat_action_set, _, mh_gat_done, mh_gat_info = mh_gat_env.reset(instance)
    #mh_gat_nb_nodes += mh_gat_info["nb_nodes"]
    #mh_gat_time += mh_gat_info["time"]
    #while not mh_gat_done:
    #    with torch.no_grad():
    #        graph = HeteroData()
    #        #node features
    #        graph['constraint'].x = torch.FloatTensor(mh_gat_observation.row_features).to(DEVICE)
    #        graph['variable'].x = torch.FloatTensor(mh_gat_observation.variable_features).to(DEVICE)
    #        #edge indices
    #        graph['variable', 'participates', 'constraint'].edge_index = torch.stack([torch.LongTensor(mh_gat_observation.edge_features.indices.astype(np.int32)[1]), torch.LongTensor(mh_gat_observation.edge_features.indices.astype(np.int32)[0])], dim=0).to(DEVICE)
    #        graph['constraint', 'contains', 'variable'].edge_index = torch.LongTensor(mh_gat_observation.edge_features.indices.astype(np.int32)).to(DEVICE)
    #        #edge features
    #        graph['variable', 'participates', 'constraint'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation.edge_features.values, axis=-1)).to(DEVICE)
    #        graph['constraint', 'contains', 'variable'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation.edge_features.values, axis=-1)).to(DEVICE)

    #        mh_gat_logits = model_multi_head_8_alt_3(graph.x_dict, graph.edge_index_dict, graph.edge_attr_dict)
    #        mh_gat_action = mh_gat_action_set[mh_gat_logits[mh_gat_action_set.astype(np.int64)].argmax()]
    #        mh_gat_observation, mh_gat_action_set, _, mh_gat_done, mh_gat_info = mh_gat_env.step(mh_gat_action)
    #    mh_gat_nb_nodes += mh_gat_info["nb_nodes"]
    #    mh_gat_time += mh_gat_info["time"]

    # Run the Multi Head (8) GAT 2 brancher
    mh_gat_nb_nodes_b, mh_gat_time_b = 0, 0
    mh_gat_observation_b, mh_gat_action_set_b, _, mh_gat_done_b, mh_gat_info_b = mh_gat_env_b.reset(instance)
    mh_gat_nb_nodes_b += mh_gat_info_b["nb_nodes"]
    mh_gat_time_b += mh_gat_info_b["time"]
    while not mh_gat_done_b:
        with torch.no_grad():
            graph = HeteroData()
            #node features
            graph['constraint'].x = torch.FloatTensor(mh_gat_observation_b.row_features).to(DEVICE)
            graph['variable'].x = torch.FloatTensor(mh_gat_observation_b.variable_features).to(DEVICE)
            #edge indices
            graph['variable', 'participates', 'constraint'].edge_index = torch.stack([torch.LongTensor(mh_gat_observation_b.edge_features.indices.astype(np.int32)[1]), torch.LongTensor(mh_gat_observation_b.edge_features.indices.astype(np.int32)[0])], dim=0).to(DEVICE)
            graph['constraint', 'contains', 'variable'].edge_index = torch.LongTensor(mh_gat_observation_b.edge_features.indices.astype(np.int32)).to(DEVICE)
            #edge features
            graph['variable', 'participates', 'constraint'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation_b.edge_features.values, axis=-1)).to(DEVICE)
            graph['constraint', 'contains', 'variable'].edge_attr = torch.FloatTensor(np.expand_dims(mh_gat_observation_b.edge_features.values, axis=-1)).to(DEVICE)

            mh_gat_logits_b = model_multi_head_8_alt_3_2(graph.x_dict, graph.edge_index_dict, graph.edge_attr_dict)
            mh_gat_action_b = mh_gat_action_set_b[mh_gat_logits_b[mh_gat_action_set_b.astype(np.int64)].argmax()]
            mh_gat_observation_b, mh_gat_action_set_b, _, mh_gat_done_b, mh_gat_info_b = mh_gat_env_b.step(mh_gat_action_b)
        mh_gat_nb_nodes_b += mh_gat_info_b["nb_nodes"]
        mh_gat_time_b += mh_gat_info_b["time"]


    # Run SCIP's default brancher
    default_env.reset(instance)
    _, _, _, _, default_info = default_env.step({})

    # Run Strong Branching
    #fsb_env.reset(instance)
    #_, _, _, _, fsb_info = fsb_env.step({})

    # Run Most Infeasible Branching
    #mib_env.reset(instance)
    #_, _, _, _, mib_info = mib_env.step({})

    #print(f"Instance {instance_count: >3} | FSB nb nodes    {int(fsb_info['nb_nodes']): >4d}  | FSB time   {fsb_info['time']: >6.2f} ")
    #print(f"Instance {instance_count: >3} | MIB nb nodes    {int(mib_info['nb_nodes']): >4d}  | MIB time   {mib_info['time']: >6.2f} ")
    print(f"Instance {instance_count: >3} | SCIP nb nodes    {int(default_info['nb_nodes']): >4d}  | SCIP time   {default_info['time']: >6.2f} ")
    print(f"             | GNN  nb nodes    {int(nb_nodes): >4d}  | GNN  time   {time: >6.2f} ")
    #print(f"             | GAT  nb nodes    {int(gat_nb_nodes): >4d}  | GAT  time   {gat_time: >6.2f} ")
    #print(f"             | MH GAT(2) nb nodes {int(mh_gat_nb_nodes_2): >4d}  | MH GAT(2) time   {mh_gat_time_2: >6.2f} ")
    #print(f"             | MH GAT(4) nb nodes {int(mh_gat_nb_nodes_4): >4d}  | MH GAT(4) time   {mh_gat_time_4: >6.2f} ")
    #print(f"             | MH GAT(8) nb nodes {int(mh_gat_nb_nodes): >4d}  | MH GAT(8) time   {mh_gat_time: >6.2f} ")
    print(f"             | MH GAT(8) nb nodes {int(mh_gat_nb_nodes_b): >4d}  | MH GAT(8) time   {mh_gat_time_b: >6.2f} ")

Instance   0 | SCIP nb nodes    5937  | SCIP time    87.22 
             | GNN  nb nodes    13123  | GNN  time   173.32 
             | MH GAT(8) nb nodes 8549  | MH GAT(8) time   126.04 
Instance   1 | SCIP nb nodes    18970  | SCIP time   247.08 
             | GNN  nb nodes    34219  | GNN  time   386.30 
             | MH GAT(8) nb nodes 42213  | MH GAT(8) time   557.08 
Instance   2 | SCIP nb nodes    5680  | SCIP time    89.89 
             | GNN  nb nodes    9843  | GNN  time   112.97 
             | MH GAT(8) nb nodes 16052  | MH GAT(8) time   205.17 
Instance   3 | SCIP nb nodes       1  | SCIP time     1.30 
             | GNN  nb nodes    231154  | GNN  time   3022.01 
             | MH GAT(8) nb nodes 144982  | MH GAT(8) time   2420.59 
